# Joint intent classification + slot filling on MASSIVE 1.0 en-US
## A systematic comparison of baseline models 
**Joint** models for MASSIVE en-US: 
- Classical HMM
- Triangular CRF
- Window MLP
- TextCNN
- ID-CNN
- BiGRU
- BiLSTM
- Transformer
- CNN -> BiGRU
- CNN -> Transformer
- 17 BiLSTM-based decoder and interaction variants. 

It crosses them with seven static word-vector sources and then adds techniques in isolation and in combination. The best configuration becomes the final system, which is frozen before a single test evaluation.

**Self-contained:**
- The first cell installs the Python packages.
- MASSIVE 1.0 (40 MB) and the six word-vector files (about 6.8 GB) are downloaded automatically the first time they are
  needed. Downloads are resumable: after a network failure, re-running the cell continues where it stopped.
- Every training run is cached in `~/.cache/sc4002/runs/` (a checkpoint plus a JSON summary), with the trimmed vector
  tables and the dataset alongside, outside the project folder so OneDrive cannot evict them. A re-run reloads cached
  runs instead of training again; a missing run is trained in place, one at a time.
- It runs on Apple's MPS, an NVIDIA GPU (CUDA) or the CPU, whichever is available.

### Status at a glance (5 November 2026)

| Part | Status |
|---|---|
| Data, preprocessing, train-only vocabularies, 7 word-vector tables | ✅ done and checked |
| Shared metric with unit tests; CRF and semi-CRF checked against brute force | ✅ |
| Model zoo: 9 family representatives, joint HMM, 17 BiLSTM variants | ✅ all memorise 64 utterances, except V15 (capsules) on intents |
| Training, caching, parameter-count assert, background runner | ✅ |
| **P0** recipe calibration (54 runs) | 🟡 **49 / 54 done**; HMM smoothing and `seqeval` cross-check pending |
| P1a baseline × embedding grid (189 runs) | ⏸ code written and tested on a mock grid; not run |
| P1b variants (102 runs + HMM) | ⏸ code written; not run |
| P2–P7 (fine-tuning, techniques, extensions, factorial, ensembles, final system, test) | ⏸ planned; code not written yet |

Section 15 collects every result so far and what is left.

### The experiment plan

**Main goal.** Compare and contrast the baseline model families **systematically and fairly**, then turn what the
comparison shows into the best possible final system, which is evaluated **once** on test.

**How the phases fit together.** Each phase answers one question and hands its answer to the next. The early phases are
broad (many models, no add-ons); the later ones are narrow and deep (one model, many refinements).

```
P0   calibrate    give every family a fair training recipe
 │
P1a  compare      9 model families × 7 word-vector sources          ← the core comparison
P1b  compare      17 BiLSTM decoder and interaction variants
P2   compare      frozen vs fine-tuned word vectors
P3   compare      4 techniques on every model, alone and together
 │
P4   refine       extensions on the single best model
P5   combine      which techniques work best together
P6   ensemble     spend the remaining parameter budget
 │
P7   finalise     retrain, reload check, ONE test evaluation
```

**The protocol every phase follows**

| Choice | Setting | Why |
|---|---|---|
| Same pipeline | Same data, train-only vocabulary, metric, batch size 32, at most 80 epochs, early stopping on dev (patience 10) | Differences then come from the models, not from the plumbing |
| Seeds | 0, 1 and 2 for every configuration | Seed-to-seed noise (about 0.3–0.6 points) is as large as many effects. Shared seeds pair the comparisons, and 3 runs give a mean and a spread |
| Selection score | Dev combined = (intent accuracy + slot F1) / 2 | Both metrics are graded equally; exact match is reported as a diagnostic |
| Keep rule | A change is adopted only if the 3-seed mean combined score rises by ≥ 0.3 and neither metric falls by > 0.3 | 0.3 is about the standard error of a 3-seed mean, so smaller gains cannot be told apart from noise. It is looser than Gilbert's rule (new mean − std > old mean + std), which rejected a configuration that was better on every metric |
| Significance | Paired bootstrap over the 2,033 dev utterances, 95% interval | Says whether two configurations really differ or are tied |
| Fairness | Equal tuning budget per family (P0), matched layer widths, every pretrained table rescaled to the same scale | No family wins through extra tuning, more width or larger input values |
| Data discipline | Dev for every decision; test used once, in P7 | Keeps the final test score an unbiased estimate |
| Budget | ≤ 20M parameters for the whole system, frozen vectors and every ensemble member included | The hard rule; only ensembles (P6) come near it |

**Summary**

| Phase | Question it answers | Runs | Feeds into | Status |
|---|---|---|---|---|
| P0 | What recipe should each family train with? | 54 + 4 HMM fits | every later phase | 49 / 54 done |
| P1a | Which family is best, which word vectors are best, and does one depend on the other? | 189 | P1b, P2, P3 | not started |
| P1b | Which slot decoder and which intent–slot interaction help a BiLSTM? | 102 + HMM | P3, P4 | not started |
| P2 | Should the word vectors be fine-tuned? | 81 | P3 | planned |
| P3 | Which techniques help which family, alone and together? | 171 | P4 | planned |
| P4 | What else improves the best model? | ~39 | P5 | planned |
| P5 | Which combination of the strongest techniques is best? | ~72 | P6 | planned |
| P6 | Does an ensemble beat a single model within 20M parameters? | ~10 | P7 | planned |
| P7 | How well does the frozen final system do on unseen test data? | retrain + 1 test pass | the report | planned |

#### P0: Calibrate (a fair recipe for every family)

**Goal.** Fix one training recipe per model family (learning rate plus dropout, or weight decay for the TriCRF) before
any comparison, and prove that the machinery works.

**Why.** Architectures prefer different settings: a Transformer trained from scratch needs a lower learning rate than a
CNN. With one shared recipe, the comparison would partly measure which family happens to suit that recipe, and Melis
et al. (2018) showed that rankings can flip once each model is tuned. Giving every family the same small tuning budget
removes this bias.

**Design.**
- Checks first (section 9): metric unit tests, CRF brute force, padding invariance, memorising 64 utterances, and a
  CPU vs GPU speed benchmark.
- Then 9 families × 6 recipes × seed 0 = **54 runs**, plus the HMM's smoothing k ∈ {0.01, 0.1, 0.5, 1} (4 fits of a
  few seconds each).

| Family | Recipe grid (6 each) |
|---|---|
| Window, TextCNN, ID-CNN, BiGRU, BiLSTM, CNN→BiGRU, CNN→attention | learning rate {5e-4, 1e-3, 2e-3} × dropout {0.2, 0.4} |
| Transformer | learning rate {2.5e-4, 5e-4, 1e-3} × dropout {0.1, 0.2}, with warmup (unstable at high learning rates) |
| TriCRF | learning rate {1e-3, 3e-3, 1e-2} × weight decay {1e-5, 1e-4} (a linear model: no dropout) |

**Design choices.**
- **Only the optimisation settings are tuned.** They are what differs between architectures. Everything a later phase
  tests (word vectors, CRF, techniques, loss weight) stays at its starting setting, so its effect cannot leak into the
  recipe.
- **Frozen GloVe 6B, per-token softmax, λ = 0.5.** These are the conditions of most of P1a and the vectors Gilbert used,
  so P0 results line up with both.
- **One seed per recipe.** The grid steps are a factor of 2 apart, so a clearly worse recipe stands out above seed noise.
  Every conclusion that matters uses 3 seeds later.
- **Selection fixed in advance.** The best dev combined score wins; recipes within 0.3 points count as tied, and the
  one nearest the family's standard default wins the tie.

**Output → next phase.** `configs/recipes.json`. Every later phase uses each family's recipe unchanged.

**Status.** 49 of 54 runs done (results in section 10).

#### P1a: The core comparison (9 families × 7 word-vector sources)

**Goal.** Rank the model families fairly, measure how much the word vectors matter, and test whether the best vectors
depend on the family.

**Why.** This is the main question of the study. Crossing every family with every vector source (instead of testing
them one after another) separates three things: the family effect, the vector effect, and their interaction. A
step-by-step ladder would make each conclusion depend on the order in which things were tried.

**Design.** 9 family representatives (TriCRF, window MLP, TextCNN, ID-CNN, BiGRU, BiLSTM, Transformer, CNN→BiGRU,
CNN→attention) × 7 sources (random, GloVe 6B, GloVe 42B, fastText wiki, fastText crawl, word2vec, Komninos) × 3 seeds
= **189 runs**, each family with its P0 recipe.

**Design choices.**
- **Pretrained vectors are frozen** here, so the comparison measures vector quality alone; fine-tuning is P2's
  question. `random` is trained from scratch as the control (for the TriCRF it means sparse features only).
- **The seven sources form planned contrasts:** pretrained vs random, GloVe vs fastText (algorithm) × Wikipedia vs web
  (corpus), and window vs dependency contexts (word2vec vs Komninos).
- **No add-ons yet** (no CRF, character features, word dropout or label smoothing), so the families are compared on
  their architecture alone; P3 adds the techniques.

**Analyses.** Tables and heatmaps; a two-way ANOVA (how much of the score each factor explains); Kendall's τ and W (do
families rank the vectors the same way?); the vector contrasts; a ranking with bootstrap ties; size and time (Pareto);
learning curves; error profiles (boundary vs type errors, recall on unseen words).

**Output → next phase.** Each family's best source; the BiLSTM's best source for P1b; the top-3 sources for P2.

**Status.** Not started; code ready and tested on a mock grid (section 12).

#### P1b: BiLSTM variants (17 decoders and interaction mechanisms)

**Goal.** Holding one strong encoder fixed, find which slot decoder and which intent–slot interaction mechanism helps.

**Why.** The literature proposes many joint-modelling mechanisms (CRFs, span decoders, slot-gates, co-attention,
capsules, graphs …), but each paper uses its own encoder, vectors and tuning, so their gains cannot be compared directly.
Putting all of them on the same BiLSTM, with the same recipe and seeds, isolates the effect of the mechanism itself.

**Design.** 17 variants × 2 sources (the BiLSTM's best from P1a, and random) × 3 seeds = **102 runs**, plus the joint HMM
(1 deterministic fit).
- **Decoders V1–V8:** CRF with BIO / BIOES / IO tags, SlotRefine, boundary → type, biaffine spans, semi-Markov CRF,
  pointer decoder.
- **Interactions V9–V17:** partial sharing, slot-gate, stack-propagation, SF-ID, CM-Net, co-interactive attention,
  capsules (⚠ flagged: cannot memorise intents), graph attention, intent–tag compatibility.

**Design choices.**
- **Every variant inherits the BiLSTM's P0 recipe.** That puts all of them on equal footing. The limitation: a variant
  that would prefer a different learning rate is slightly disadvantaged.
- **Two sources** show whether a mechanism matters more when the model has no pretrained knowledge (random vectors).
- **Loss weight λ = 0.7 for sequence-level slot losses** (CRF, semi-CRF), because they sum over tokens and are about
  7× larger than a per-token loss.
- **Each variant is compared with the plain BiLSTM** (from P1a, same source and seeds), with a bootstrap interval.

**Output → next phase.** The best decoder and interaction mechanisms are carried into P3 and P4.

**Status.** Not started; code ready (section 13).

#### P2: Frozen vs fine-tuned word vectors

**Goal.** Decide, per family, whether the word vectors should stay frozen or be adapted during training.

**Why.** Frozen vectors keep their pretrained geometry, which protects rare words. Fine-tuning lets frequent task words
("alarm", "play") move to better positions for this task. The usual argument for freezing (seen and unseen words must
stay in one shared space) does not apply here, because unseen words map to `<UNK>` anyway. So this is an empirical
question.

**Design.** 9 representatives × the top-3 pretrained sources from P1a × fine-tuned × 3 seeds = **81 runs**, each
compared with its frozen P1a counterpart (same seeds).

**Design choices.**
- **The vectors learn at 0.1 × the model's learning rate,** enough to adapt without wiping out what pretraining learnt.
- **Only the top-3 sources** are fine-tuned, to save compute; weaker sources are unlikely to overtake after
  fine-tuning.

**Output → next phase.** For each family, frozen or fine-tuned vectors, used from P3 on.

**Status.** Planned; code not written yet.

#### P3: Techniques, alone and together, on every model

**Goal.** Measure what each standard technique adds to each model, and whether adding all of them changes the ranking.

**Why.** P1–P2 compare bare architectures, but published systems almost always add these techniques, and the gains can
be large (a CRF added about 4 slot-F1 points to Gilbert's Transformer). A family that looks weak bare might win once
equipped, so the ranking has to be re-checked.

**Design.**
- 8 neural representatives × 5 arms × 3 seeds = 120 runs (the TriCRF already has a CRF and its own features).
- 17 variants × "all applicable" × 3 seeds = 51 runs.
- **171 runs** in total, each model at its best P1a/P2 setting.

| Arm | What it adds | Targets |
|---|---|---|
| +CRF | Whole-sequence decoding with illegal tag moves blocked (λ = 0.7) | Broken or inconsistent slot spans |
| +char-CNN | Character features for every word | Unseen and misspelt words |
| +word dropout | Rare words randomly hidden as `<UNK>` during training (α = 0.75, which hides 3.56% of training words, matching the 3.32% unseen words in dev) | A well-trained "unknown word" vector |
| +label smoothing | Softened intent targets (ε = 0.1) | Overconfidence and noisy labels |
| +all four | Everything together | Do the gains add up? |

**Design choices.**
- **Each technique alone, plus all four together,** not all 16 combinations. That gives every isolated effect and the
  full package at a fraction of the cost; interactions are measured properly in P5, on the best model only.

**Output → next phase.** A model × technique gain matrix, and the best overall configuration, which goes to P4.

**Status.** Planned; code not written yet.

#### P4: Extensions on the best model

**Goal.** Push the single strongest configuration further with targeted improvements.

**Why.** By now the remaining errors are known from the error profiles: unseen words in slots, span boundaries, and
confusable intents (e.g. `calendar_query` vs `calendar_set`). Each extension targets one of them.

**Design.** About **39 runs** on the best configuration from P3, 3 seeds each:

| Extension | Targets |
|---|---|
| Character BiLSTM, train-only subword pieces (BPE), word-shape flags, possessive backoff ("X's" → "X") | Unseen words |
| Word-dropout strength α ∈ {0.25, 1}; loss weight λ ∈ {0.5, 0.85} | Tuning the strongest knobs |
| Supervised contrastive loss | Confusable intents |
| Boundary and type auxiliary losses | Span boundaries |
| R-Drop, weight averaging (EMA) | Training noise and overfitting |
| Self-distillation (our own model as teacher, train data only) | Smoother targets |

**Design choices.**
- **Only on the best model,** because testing every extension on every family would cost hundreds of runs for
  conclusions about models that are no longer in contention.
- **Each extension is tested against the current best** and kept only if it passes the keep rule.

**Output → next phase.** The five strongest techniques, which go to P5.

**Status.** Planned; code not written yet.

#### P5: Combination (which techniques work best together)

**Goal.** Find the best combination of the strongest techniques, and measure whether they reinforce or cancel each other.

**Why.** Testing one change at a time misses interactions: two techniques can fix the same errors (so together they add
little) or complement each other (so together they add more than the sum).

**Design.**
- A 2^(5−1) fractional factorial: 16 combinations of 5 on/off techniques × 3 seeds = 48 runs.
- Then about 24 runs of random search over the continuous settings (learning rate, dropout, word-dropout α, λ).
- About **72 runs** in total.

**Design choices.**
- **Half of the full 32-combination design.** It still estimates every main effect and every two-technique
  interaction cleanly (resolution V), at half the cost.
- **Random search instead of a grid** for the continuous settings, because it covers each important setting better
  when only a few of them matter (Bergstra & Bengio 2012).

**Output → next phase.** The final single-model configuration.

**Status.** Planned; code not written yet.

#### P6: Ensembles within the 20M budget

**Goal.** Use the remaining parameter budget to reduce prediction variance by combining several models.

**Why.** Every single model is far below the 20M cap (8M at most), and averaging several members cancels part of their
seed-to-seed noise. Members that differ more (different families or different word vectors) make less correlated
errors, so they may help more than extra seeds of one model.

**Design.** About **10 configurations**, compared at the same number of members:
- several seeds of the best configuration;
- the best model of several different families;
- the best model with different word vectors.

**Design choices.**
- **One combination rule:** average the intent probabilities and sum the members' CRF scores, then decode once. Summing
  CRF scores is exact (a product of experts) and keeps every output a valid tag sequence.
- **Member lists are fixed in advance.** Picking the luckiest seeds on dev would inflate the dev score without helping
  on test (the winner's curse).
- **The total parameter count of all members together is asserted ≤ 20M.**

**Output → next phase.** The final system: the best single model or the best ensemble.

**Status.** Planned; code not written yet.

#### P7: Final system and the single test evaluation

**Goal.** Produce the deliverable and an unbiased estimate of how the final system performs on unseen data.

**Why.** Everything up to here was chosen on dev, so dev scores are slightly optimistic. A fresh retrain and one untouched
test evaluation give the honest number.

**Design.**
1. Freeze the final configuration.
2. Retrain it with **fresh seeds (100+)**, logging per-epoch train and dev loss, dev metrics, elapsed time and learning
   curves.
3. Save the best-dev checkpoint and print its epoch, score and filename.
4. Reload the checkpoint and assert that it reproduces the logged dev scores exactly.
5. Evaluate **test once**: the final system, plus each family's best configuration for the comparison table.
6. Error analysis: per-slot-type F1, intent confusions, and recall on spans with unseen words.

**Design choices.**
- **Fresh seeds,** so the final dev score is not inflated by having picked the best seeds earlier.
- **The reload check runs on the CPU in evaluation mode,** where computation is deterministic, so "reloaded = logged"
  can be asserted exactly.
- **No decision follows the test numbers.** Nothing is changed after the test cell.

**Status.** Planned; code not written yet.

## 1. Setup

The first cell installs the packages (PyTorch, NumPy, SciPy, Matplotlib, `seqeval`); it does nothing if they are
already present. The second holds imports, paths and constants used everywhere, plus three helpers:
- `set_seed` seeds Python, NumPy and PyTorch;
- `pick_device` maps the GPU choice to Apple's MPS, else CUDA, else the CPU;
- `download` fetches a file once, resumably, and is used for the dataset and every vector file.

Large files live outside the project folder and outside OneDrive (`~/.cache/sc4002/`): the raw vector files, the trimmed
vector tables, the extracted dataset and the run cache. `CODE_VERSION` is part of every run's cache key, so bumping it
retrains everything after a change that affects results.

In [ ]:
%pip install -q torch numpy scipy matplotlib seqeval

In [1]:
import collections, copy, gzip, hashlib, itertools, json, math, os, random, re, sys, tarfile, time, urllib.error, urllib.request, zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DATA_DIR = "data"                                   # git-ignored: the downloaded MASSIVE archive
RUNS_DIR = "runs"                                   # git-ignored: final checkpoints and result tables
CACHE_DIR = os.path.expanduser("~/.cache/sc4002")   # outside the repo and OneDrive
VEC_DIR = os.path.join(CACHE_DIR, "vectors")        # raw vector files, downloaded on first use
RUN_CACHE = os.path.join(CACHE_DIR, "runs")         # one checkpoint + one JSON log per sweep run
EMB_CACHE = os.path.join(CACHE_DIR, "emb")          # trimmed vector tables (OneDrive can evict files that runs read)
for _d in (DATA_DIR, RUNS_DIR, RUN_CACHE, EMB_CACHE, "configs"):
    os.makedirs(_d, exist_ok=True)

MASSIVE_URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz"
MAX_PARAMS = 20_000_000
SEEDS = [0, 1, 2]
PAD, UNK, IGNORE = 0, 1, -100
CODE_VERSION = "2026-11-04b"   # bump to invalidate cached runs after a change that affects training


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def pick_device(name):
    """'mps' means "the GPU": Apple's MPS when available, else CUDA, else the CPU."""
    if name == "mps":
        if torch.backends.mps.is_available():
            return torch.device("mps")
        if torch.cuda.is_available():
            return torch.device("cuda")
    return torch.device("cpu")


def download(url, dest, retries=5):
    """Download `url` to `dest` unless it is already there. Resumable: an interrupted download continues from where it
    stopped (HTTP Range), so re-running the cell after a network failure picks up the rest."""
    if os.path.exists(dest):
        return dest
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    part = dest + ".part"
    for attempt in range(1, retries + 1):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        request = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
        try:
            with urllib.request.urlopen(request, timeout=60) as r:
                if have and r.status != 206:                  # the server ignored the range: start again
                    have = 0
                size = r.headers.get("Content-Length")
                total = have + int(size) if size else None
                print(f"downloading {os.path.basename(dest)}" + (f" ({total / 1e9:.2f} GB)" if total else "")
                      + (f", resuming at {have / 1e9:.2f} GB" if have else ""), flush=True)
                with open(part, "ab" if have else "wb") as f:
                    while chunk := r.read(1 << 20):
                        f.write(chunk)
            if total is None or os.path.getsize(part) == total:
                os.replace(part, dest)
                return dest
        except urllib.error.HTTPError as e:
            if e.code == 416 and have:                         # nothing left to fetch: the partial file is complete
                os.replace(part, dest)
                return dest
            print(f"  attempt {attempt}/{retries} failed: {e}", flush=True)
        except (urllib.error.URLError, OSError) as e:
            print(f"  attempt {attempt}/{retries} failed: {e}", flush=True)
        time.sleep(5)
    raise RuntimeError(f"could not download {url}; re-run the cell to resume")

`show_table` prints a Markdown table: rendered inside Jupyter, plain text elsewhere. Every results cell uses it.

In [2]:
def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)] + ["| " + " | ".join(map(str, r)) + " |" for r in rows]
    if "ipykernel" in sys.modules:                            # rendered table inside Jupyter, plain text elsewhere
        from IPython.display import Markdown, display
        display(Markdown("\n".join(lines)))
    else:
        print("\n".join(lines))

**Recorded environment:** Python 3.11.18, PyTorch 2.10.0, NumPy 2.4.6, SciPy 1.17.1, Matplotlib 3.11.2,
seqeval 1.2.2; MPS available (Apple M4, 16 GB RAM).

## 2. Data

### Pre-processing 1: Download, load and assert the official splits

Only `1.0/data/en-US.jsonl` from the official release is used. The tarball (39,500,415 bytes) is downloaded to `data/`
and the one file is extracted to `~/.cache/sc4002/massive/`; both steps are skipped when already done. Every record keeps its official `partition`; nothing is re-split, added or dropped.

In [3]:
def ensure_massive():
    """Return the path of 1.0/data/en-US.jsonl, downloading and extracting it if needed (extracted outside OneDrive)."""
    jsonl = os.path.join(CACHE_DIR, "massive", "1.0", "data", "en-US.jsonl")
    if not os.path.exists(jsonl):
        archive = download(MASSIVE_URL, os.path.join(DATA_DIR, os.path.basename(MASSIVE_URL)))
        with tarfile.open(archive) as tar:
            wanted = [m for m in tar.getmembers() if m.name in ("1.0/data/en-US.jsonl", "1.0/LICENSE", "1.0/NOTICE.md")]
            tar.extractall(os.path.join(CACHE_DIR, "massive"), members=wanted, filter="data")
    return jsonl


def load_splits(path):
    splits = {"train": [], "dev": [], "test": []}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                splits[row["partition"]].append(row)
    return splits

In [4]:
DATA_PATH = ensure_massive()
splits = load_splits(DATA_PATH)
assert {k: len(v) for k, v in splits.items()} == {"train": 11514, "dev": 2033, "test": 2974}
assert sum(len(v) for v in splits.values()) == 16521
assert len({r["id"] for rows in splits.values() for r in rows}) == 16521, "record ids must be unique"
assert all(r["locale"] == "en-US" for rows in splits.values() for r in rows)

**Expected:** all asserts pass: 11,514 / 2,033 / 2,974 rows, 16,521 in total, unique ids, all `en-US`.
(Version 1.1, which `baseline_gilbert.ipynb` downloads, has identical en-US records: checked during planning.)

### Pre-processing 2: Parse each annotation into token spans

`annot_utt` marks slots inline, e.g. `wake me up at [time : five am]`. `parse_annotation` reads it left to right into
the plain text plus character offsets. `to_example` asserts that the text equals `utt` exactly and converts each slot
to a token span (type, start, end); it raises if a slot does not start and end on whitespace boundaries.

In [5]:
SLOT_RE = re.compile(r"\[([^\]\[]*?) : ([^\]\[]*?)\]")


def parse_annotation(annot_utt):
    """'wake me up at [time : five am]' -> ('wake me up at five am', [('time', 14, 21)]) with character offsets."""
    text, spans, pos = "", [], 0
    for m in SLOT_RE.finditer(annot_utt):
        text += annot_utt[pos:m.start()]
        start = len(text)
        text += m.group(2)
        spans.append((m.group(1), start, len(text)))
        pos = m.end()
    return text + annot_utt[pos:], spans


def to_example(row):
    text, char_spans = parse_annotation(row["annot_utt"])
    assert text == row["utt"], row["id"]                     # the annotation rebuilds the model input exactly
    tokens = row["utt"].split(" ")
    starts, offset = [], 0
    for tok in tokens:
        starts.append(offset)
        offset += len(tok) + 1
    ends = [s + len(t) for s, t in zip(starts, tokens)]
    spans = [(slot, starts.index(cs), ends.index(ce) + 1) for slot, cs, ce in char_spans]   # raises if not token-aligned
    return {"id": row["id"], "tokens": tokens, "spans": spans, "intent": row["intent"]}

### Pre-processing 3: Three tagging schemes

The models tag one label per token, so spans become tags and back:
- **BIO** (111 tags): `B-x` starts a span, `I-x` continues it;
- **BIOES** (221 tags): adds `E-x` for span ends and `S-x` for single-token spans;
- **IO** (56 tags): one tag per slot type. Lossless here, because no two same-type spans are ever adjacent.

`tags_to_spans` follows the conlleval convention: a stray `I-x` opens a new span. A constrained CRF never produces one.

In [7]:
def spans_to_tags(n, spans, scheme="BIO"):
    tags = ["O"] * n
    for slot, s, e in spans:
        if scheme == "IO":
            tags[s:e] = [slot] * (e - s)
        elif scheme == "BIO":
            tags[s:e] = ["B-" + slot] + ["I-" + slot] * (e - s - 1)
        elif e - s == 1:                                      # BIOES single-token span
            tags[s] = "S-" + slot
        else:
            tags[s:e] = ["B-" + slot] + ["I-" + slot] * (e - s - 2) + ["E-" + slot]
    return tags


def tags_to_spans(tags, scheme="BIO"):
    """Decode tags into (type, start, end) spans. A stray I-x opens a span (conlleval rule); a stray E-x is one token."""
    spans, cur = [], None
    for i, tag in enumerate(tags):
        if scheme == "IO":
            slot = None if tag == "O" else tag
            if cur is not None and slot == cur[0]:
                cur[2] = i + 1
                continue
            if cur is not None:
                spans.append(tuple(cur))
                cur = None
            if slot is not None:
                cur = [slot, i, i + 1]
            continue
        prefix, slot = ("O", None) if tag == "O" else (tag[0], tag[2:])
        if cur is not None and slot == cur[0] and prefix in ("I", "E"):
            cur[2] = i + 1
            if prefix == "E":
                spans.append(tuple(cur))
                cur = None
            continue
        if cur is not None:
            spans.append(tuple(cur))
            cur = None
        if prefix in ("B", "I"):
            cur = [slot, i, i + 1]
        elif prefix in ("S", "E"):
            spans.append((slot, i, i + 1))
    if cur is not None:
        spans.append(tuple(cur))
    return spans

### Pre-processing 4: Build the examples and check the round trip

Every gold span must survive spans → tags → spans under all three schemes. One assert tests the parser, the tokeniser
and the tag encoders together on every row.

In [9]:
examples = {split: [to_example(r) for r in rows] for split, rows in splits.items()}
for _scheme in ("BIO", "BIOES", "IO"):                        # every gold span survives the round trip in every scheme
    for _exs in examples.values():
        for _ex in _exs:
            assert tags_to_spans(spans_to_tags(len(_ex["tokens"]), _ex["spans"], _scheme), _scheme) == _ex["spans"], (_scheme, _ex["id"])

**Expected:** passes for all 16,521 utterances under BIO, BIOES and IO.

### Pre-processing 5: Train-only vocabularies

Words, characters, intents, slot types and tags are collected from **train only**; dev and test words that train never
saw map to `<UNK>`. `VOCAB_HASH` fingerprints the vocabulary and is part of every cache key.

In [10]:
def build_vocab(train):
    word_counts = collections.Counter(w for ex in train for w in ex["tokens"])
    words = ["<PAD>", "<UNK>"] + sorted(word_counts)
    chars = ["<PAD>", "<UNK>"] + sorted({c for w in word_counts for c in w})
    intents = sorted({ex["intent"] for ex in train})
    slot_types = sorted({slot for ex in train for slot, _, _ in ex["spans"]})
    tagsets = {"BIO": ["O"] + [f"{p}-{s}" for s in slot_types for p in "BI"],
               "BIOES": ["O"] + [f"{p}-{s}" for s in slot_types for p in "BIES"],
               "IO": ["O"] + slot_types}
    return {"word2id": {w: i for i, w in enumerate(words)}, "char2id": {c: i for i, c in enumerate(chars)},
            "intent2id": {name: i for i, name in enumerate(intents)}, "slot_types": slot_types,
            "tags": tagsets, "tag2id": {k: {t: i for i, t in enumerate(v)} for k, v in tagsets.items()},
            "word_counts": word_counts}


vocab = build_vocab(examples["train"])                        # train only: dev and test never touch the vocabularies
ID2INTENT = sorted(vocab["intent2id"], key=vocab["intent2id"].get)
assert len(vocab["intent2id"]) == 60 and len(vocab["slot_types"]) == 55
assert len(vocab["word2id"]) == 5304 and len(vocab["char2id"]) == 34
assert {k: len(v) for k, v in vocab["tags"].items()} == {"BIO": 111, "BIOES": 221, "IO": 56}
assert all(ex["intent"] in vocab["intent2id"] for split in ("dev", "test") for ex in examples[split])
VOCAB_HASH = hashlib.md5(json.dumps(sorted(vocab["word2id"].items())).encode()).hexdigest()[:8]

**Expected:** 5,304 words (5,302 train types + `<PAD>` + `<UNK>`), 34 characters, 60 intents, 55 slot types,
tags BIO 111 / BIOES 221 / IO 56, every dev and test intent seen in train, `VOCAB_HASH = e523ad6f`.

### Data profile (train and dev only)

Test is never inspected beyond its size.

In [11]:
n_tok = sum(len(ex["tokens"]) for ex in examples["train"])
print(f"train: {len(examples['train'])} utterances, {n_tok} tokens, {len(vocab['word2id']) - 2} word types, "
      f"{sum(len(ex['spans']) for ex in examples['train'])} slot spans")
dev_oov = sum(w not in vocab["word2id"] for ex in examples["dev"] for w in ex["tokens"])
print(f"dev OOV tokens: {100 * dev_oov / sum(len(ex['tokens']) for ex in examples['dev']):.2f}%   (EDA uses train and dev only)")

train: 11514 utterances, 79732 tokens, 5302 word types, 11344 slot spans
dev OOV tokens: 3.32%   (EDA uses train and dev only)


**Recorded result:**

```
train: 11514 utterances, 79732 tokens, 5302 word types, 11344 slot spans
dev OOV tokens: 3.32%   (EDA uses train and dev only)
```

Other facts measured on train and dev during planning, which shape the design:

| Fact | Implication |
|---|---|
| Mean length 6.9 tokens (train max 35) | Short sequences: a CRF's O(T·K²) cost is negligible |
| 2,537 of 5,302 train word types are singletons | Half the vocabulary gets few updates, so pretrained vectors matter most for exactly those words |
| Dev OOV: 3.32% of tokens, and 11.2% of dev slot mentions contain an OOV token | Unknown words concentrate in slots, so OOV handling is a slot-F1 issue |
| 3,755 train utterances (32.6%) and 646 dev utterances have no slots | They can only add false positives, so precision matters |
| 31 of 55 slot types put ≥ 95% of their mentions in one scenario | Intent and slots are tightly coupled, which motivates the interaction variants |
| 74.0% of dev slot mentions are (type, value) pairs seen in train | Much of slot filling is recognition plus disambiguation |
| No two same-type spans are adjacent anywhere | IO tagging is lossless |

## 3. Word vectors

### Pre-processing 6: Seven candidate sources

All are 300-d, so the choice costs the same 1,591,200 parameters (5,304 × 300) whichever source wins.

| Name | Source | Casing / corpus |
|---|---|---|
| `random` | No file: a trainable table drawn from N(0, 0.4²) (the control) | — |
| `glove6b` | GloVe 6B (the source in `baseline_gilbert.ipynb`) | uncased, Wikipedia + Gigaword |
| `glove42b` | GloVe 42B | uncased, Common Crawl |
| `ft_wiki` | fastText wiki-news-300d-1M-subword | cased, Wikipedia + news |
| `ft_crawl` | fastText crawl-300d-2M | cased, Common Crawl |
| `w2v` | word2vec GoogleNews (window contexts) | cased, news |
| `komninos` | Komninos & Manandhar (dependency-based word2vec) | lowercase, Wikipedia |

Each file is downloaded automatically, once, into `~/.cache/sc4002/vectors/` the first time its table is built
(`VECTOR_URLS` below; about 6.8 GB in total). All seven URLs were checked on 5 November 2026 to serve resumable
(HTTP Range) downloads.

In [ ]:
EMBEDDINGS = {   # name -> (file in VEC_DIR, member inside the zip or None, format)
    "glove6b": ("glove.6B.zip", "glove.6B.300d.txt", "text"),
    "glove42b": ("glove.42B.300d.zip", "glove.42B.300d.txt", "text"),
    "ft_wiki": ("wiki-news-300d-1M-subword.vec.zip", "wiki-news-300d-1M-subword.vec", "text"),
    "ft_crawl": ("crawl-300d-2M.vec.zip", "crawl-300d-2M.vec", "text"),
    "w2v": ("word2vec-google-news-300.gz", None, "w2v_bin"),
    "komninos": ("komninos_english_embeddings.gz", None, "text"),
}
VECTOR_URLS = {   # about 6.8 GB in total; each file is downloaded once, on first use, then trimmed and cached
    "glove6b": "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip",
    "glove42b": "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.42B.300d.zip",
    "ft_wiki": "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M-subword.vec.zip",
    "ft_crawl": "https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip",
    "w2v": "https://github.com/RaRe-Technologies/gensim-data/releases/download/word2vec-google-news-300/word2vec-google-news-300.gz",
    "komninos": "https://public.ukp.informatik.tu-darmstadt.de/reimers/embeddings/komninos_english_embeddings.gz",
}
EMBEDDING_NAMES = ["random"] + list(EMBEDDINGS)
EMB_DIM = 300
RANDOM_STD = 0.4

### Pre-processing 7: Stream each file once and keep only train-vocabulary rows

Each file is read line by line straight from its archive (never unzipped to disk) and only the rows the vocabulary
needs are kept. Words with an apostrophe also request their pieces (`amazon's` → `amazon` + `'s`).

In [ ]:
def split_apostrophe(word):
    """amazon's -> ('amazon', "'s"); aren't -> ('are', "n't")  (same rule as baseline_gilbert.ipynb)."""
    i = len(word) - 3 if word.endswith("n't") else word.index("'")
    return word[:i], word[i:]


def _scan_text(f, wanted):
    table = {}
    for raw in f:
        word, _, rest = raw.rstrip().partition(b" ")
        w = word.decode("utf-8", "replace")
        if w.lower() in wanted:
            vals = rest.split()
            if len(vals) == EMB_DIM:                   # also skips header lines such as "999994 300"
                table[w] = np.asarray(vals, dtype=np.float32)
    return table


def _scan_w2v_bin(f, wanted):
    table = {}
    n, d = map(int, f.readline().split())
    assert d == EMB_DIM
    vb, buf, pos = 4 * d, f.read(1 << 24), 0
    for _ in range(n):
        sp = buf.find(b" ", pos)
        while sp == -1 or len(buf) - sp - 1 < vb:
            more = f.read(1 << 24)
            if not more:
                break
            buf, pos = buf[pos:] + more, 0
            sp = buf.find(b" ")
        w = buf[pos:sp].lstrip(b"\n").decode("utf-8", "replace")
        if w.lower() in wanted:
            table[w] = np.frombuffer(buf[sp + 1:sp + 1 + vb], dtype="<f4").copy()
        pos = sp + 1 + vb
    return table

### Pre-processing 8: Build each table

The lookup follows `baseline_gilbert.ipynb`: the exact form, then Capitalised, then UPPER (so lowercase text can use cased
sources). A still-missing word with an apostrophe takes the mean of its pieces. Remaining words get N(0, σ_table), `<PAD>`
is zero and `<UNK>` is the table's mean vector. Each trimmed table (about 6 MB) is cached in `~/.cache/sc4002/emb/`, so
the raw file is downloaded and scanned only when this vocabulary's table is not cached yet.

In [ ]:
def embedding_matrix(name):
    """(matrix [V, 300] in word2id order, coverage info) for one source; cached in ~/.cache/sc4002/emb/."""
    if name == "random":
        return None, {"name": "random"}
    cache = os.path.join(EMB_CACHE, f"{name}_{VOCAB_HASH}.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=False)
        return z["matrix"], json.loads(str(z["info"]))
    word2id = vocab["word2id"]
    wanted = {w for w in word2id if w not in ("<PAD>", "<UNK>")}
    wanted |= {p for w in list(wanted) if "'" in w for p in split_apostrophe(w)}
    fname, member, fmt = EMBEDDINGS[name]
    path = download(VECTOR_URLS[name], os.path.join(VEC_DIR, fname))
    with (zipfile.ZipFile(path).open(member) if member else gzip.open(path, "rb")) as f:
        table = _scan_text(f, wanted) if fmt == "text" else _scan_w2v_bin(f, wanted)

    def find(w):
        for v in (w, w.capitalize(), w.upper()):
            if v in table:
                return table[v]
        return None

    found = np.stack(list(table.values()))
    rng = np.random.default_rng(0)
    matrix = np.zeros((len(word2id), EMB_DIM), dtype=np.float32)
    missing = []
    for w, i in word2id.items():
        if w == "<PAD>":
            continue
        if w == "<UNK>":
            matrix[i] = found.mean(0)
            continue
        vec = find(w)
        if vec is None and "'" in w:
            pieces = [find(p) for p in split_apostrophe(w)]
            if all(p is not None for p in pieces):
                vec = np.mean(pieces, axis=0)
        if vec is None:
            vec = rng.normal(0.0, found.std(), EMB_DIM)
            missing.append(w)
        matrix[i] = vec
    counts = vocab["word_counts"]
    info = {"name": name, "types": len(word2id) - 2, "missing_types": len(missing),
            "missing_pct": 100 * len(missing) / (len(word2id) - 2),
            "token_coverage_pct": 100 * (1 - sum(counts[w] for w in missing) / sum(counts.values())),
            "table_std": float(found.std()), "missing_examples": missing[:10]}
    np.savez(cache, matrix=matrix, info=json.dumps(info))
    return matrix, info

### Coverage check

On a first run this cell downloads the six vector files and builds their tables (the slowest data step: about 6.8 GB and
a few minutes per file). Later runs read the cached tables in seconds.

In [ ]:
show_table(["source", "train word types missing", "token coverage", "raw std (rescaled to 0.4)"],
           [[n, f"{i['missing_types']} ({i['missing_pct']:.2f}%)", f"{i['token_coverage_pct']:.2f}%", f"{i['table_std']:.3f}"]
            for n, i in ((n, embedding_matrix(n)[1]) for n in EMBEDDINGS)])

**Recorded result:**

| Source | Train word types missing | Token coverage | Raw std (rescaled to 0.4 in section 7) |
|---|---|---|---|
| glove6b | 184 (3.47%) | 99.65% | 0.373 |
| glove42b | 79 (1.49%) | 99.88% | 0.396 |
| ft_wiki | 140 (2.64%) | 99.67% | 0.027 |
| ft_crawl | 89 (1.68%) | 99.86% | 0.295 |
| w2v | 142 (2.68%) | 99.70% | 0.183 |
| komninos | 253 (4.77%) | 99.53% | 0.281 |

The GloVe 6B and fastText wiki-news figures match `baseline_gilbert.ipynb`. The Common Crawl sources cover the most
words. The raw scales differ by 14× (0.027 vs 0.396), so every pretrained table is rescaled to the random table's
std of 0.4 before training; otherwise a frozen-vector comparison would partly compare input scales.

## 4. Encoding and batching

### Pre-processing 9: Encode every utterance

Each utterance becomes word ids, character ids (at most 20 per word), tag ids and its intent id. Encoded splits are cached
per tag scheme.

In [ ]:
MAX_CHARS = 20


def encode(ex, scheme="BIO"):
    w2i, c2i, t2i = vocab["word2id"], vocab["char2id"], vocab["tag2id"][scheme]
    return {"words": [w2i.get(w, UNK) for w in ex["tokens"]],
            "chars": [[c2i.get(c, UNK) for c in w[:MAX_CHARS]] for w in ex["tokens"]],
            "tags": [t2i[t] for t in spans_to_tags(len(ex["tokens"]), ex["spans"], scheme)],
            "intent": vocab["intent2id"][ex["intent"]]}


_ENCODED = {}


def encoded(split, scheme="BIO"):
    if (split, scheme) not in _ENCODED:
        _ENCODED[(split, scheme)] = [encode(ex, scheme) for ex in examples[split]]
    return _ENCODED[(split, scheme)]

### Pre-processing 10: Pad a batch

`collate` pads to the batch's longest utterance and builds the mask every later layer uses (True = real token). Tags are
padded with −100, never 0, because tag id 0 is `O`. Every batch has at least 2 positions: on Apple's MPS backend the
convolution weight-gradient kernel aborts when every utterance in a batch is one token long (130 train utterances are).
The extra position is padding, masked everywhere, so no output changes (checked in section 9).

In [ ]:
def collate(batch):
    # At least 2 positions: on MPS, the convolution weight-gradient kernel aborts (a Metal layout assertion) when every
    # utterance in a batch is one token long. The extra position is PAD and masked everywhere, so no output changes
    # (the padding-invariance smoke test checks exactly this).
    B, T = len(batch), max(2, max(len(x["words"]) for x in batch))
    C = max(len(c) for x in batch for c in x["chars"])
    words = torch.full((B, T), PAD, dtype=torch.long)
    chars = torch.full((B, T, C), PAD, dtype=torch.long)
    tags = torch.full((B, T), IGNORE, dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x["words"])
        words[i, :n] = torch.tensor(x["words"])
        tags[i, :n] = torch.tensor(x["tags"])
        for j, cs in enumerate(x["chars"]):
            chars[i, j, :len(cs)] = torch.tensor(cs)
    out = {"words": words, "chars": chars, "tags": tags, "mask": words != PAD,
           "intents": torch.tensor([x["intent"] for x in batch])}
    if "tok_feats" in batch[0]:                       # sparse features of the classical TriCRF
        F_ = max(len(f) for x in batch for f in x["tok_feats"])
        U = max(len(x["utt_feats"]) for x in batch)
        tf = torch.zeros((B, T, F_), dtype=torch.long)
        uf = torch.zeros((B, U), dtype=torch.long)
        for i, x in enumerate(batch):
            for j, f in enumerate(x["tok_feats"]):
                tf[i, j, :len(f)] = torch.tensor(f)
            uf[i, :len(x["utt_feats"])] = torch.tensor(x["utt_feats"])
        out["tok_feats"], out["utt_feats"] = tf, uf
    return out

### Pre-processing 11: Length-bucketed batches

Utterances of similar length share a batch, so padding stays small; batches are shuffled with a seed per epoch.

In [ ]:
def batch_indices(data, batch_size, shuffle, seed=0):
    if not shuffle:
        idx = sorted(range(len(data)), key=lambda k: len(data[k]["words"]))
        return [idx[i:i + batch_size] for i in range(0, len(idx), batch_size)]
    rng = random.Random(seed)
    idx = list(range(len(data)))
    rng.shuffle(idx)
    pool = batch_size * 50
    out = []
    for i in range(0, len(idx), pool):
        chunk = sorted(idx[i:i + pool], key=lambda k: len(data[k]["words"]))
        out += [chunk[j:j + batch_size] for j in range(0, len(chunk), batch_size)]
    rng.shuffle(out)
    return out


def to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

## 5. The shared metric

One implementation scores every model:
- **Intent accuracy** = 100 × correct / total.
- **Slot micro-F1:** for each utterance, gold and predicted slots are **multisets** of (slot type, span text) pairs.
  TP / FP / FN are pooled over all utterances, and F1 = 100 × 2TP / (2TP + FP + FN), which is 0.00 when the
  denominator is 0. A boundary error costs one FP and one FN; there is no partial credit.
- **Exact match** (intent right and the slot multiset identical) is a diagnostic.
- **Combined** = (intent accuracy + slot F1) / 2 is the selection score.

In [ ]:
def slot_pairs(tokens, spans):
    return collections.Counter((slot, " ".join(tokens[s:e])) for slot, s, e in spans)


def score_predictions(exs, pred_intents, pred_spans):
    tp = fp = fn = correct = exact = 0
    for ex, pi, ps in zip(exs, pred_intents, pred_spans, strict=True):
        gold, pred = slot_pairs(ex["tokens"], ex["spans"]), slot_pairs(ex["tokens"], ps)
        hit = sum((gold & pred).values())
        tp += hit
        fp += sum(pred.values()) - hit
        fn += sum(gold.values()) - hit
        ok = pi == ex["intent"]
        correct += ok
        exact += ok and gold == pred
    n = len(exs)
    f1 = 100.0 * 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    acc = 100.0 * correct / n
    return {"intent_acc": acc, "slot_f1": f1, "exact_match": 100.0 * exact / n, "combined": (acc + f1) / 2,
            "tp": tp, "fp": fp, "fn": fn, "n_correct_intent": correct, "n": n}

## 6. The model zoo

Every model is **one joint network with one loss** that predicts both the intent and the slots.

| ID | Family | Context mechanism | Where |
|---|---|---|---|
| R0 | Joint intent-conditioned HMM | Counts; Markov tags under each intent | Step 5 |
| R1 | Joint triangular CRF | Sparse features (+ optional static vectors); intent and tags normalised together | Step 5 |
| R2 | Window MLP | Each token sees only its ±1 neighbours | Step 3 |
| R3 | TextCNN | Convolutions of widths 1/3/5, 2 layers | Step 3 |
| R4 | ID-CNN | Dilated convolutions (1, 2, 4), block iterated 4× with shared weights | Step 3 |
| R5 / R6 | BiGRU / BiLSTM | 2 layers × 256 per direction | Step 3 |
| R7 | Transformer | 3 pre-LN layers, d 256, relative-position attention (Gilbert's encoder) | Step 3 |
| R8 | CNN → BiGRU | One width-3 convolution, then a 1-layer BiGRU | Step 3 |
| R9 | CNN → attention | One width-3 convolution, then 2 Transformer layers | Step 3 |
| V1–V17 | BiLSTM variants | Same BiLSTM; different slot decoder (V1–V8) or intent–slot interaction (V9–V17) | Step 6 |

Neural families share one recipe: `Embedder → encoder → heads`, where the intent head max-pools the token states and the
slot head scores each token's tags (decoded by softmax, or by the CRF when that arm is switched on).

### Step 1: A linear-chain CRF with hard transition constraints

The CRF scores a whole tag sequence: s(y) = π[y₁] + Σₜ E[t, yₜ] + Σₜ A[yₜ₋₁, yₜ] + η[y_T]. Illegal moves
(e.g. O → I-time, or starting on I-) are held at −10⁴, so Viterbi can never output an invalid sequence.
`log_partition` is the forward algorithm (exp-space matrix products: 22 ms per TriCRF step instead of 218–312 ms);
`viterbi` returns the best path; `nll` is the training loss.

In [ ]:
NEG = -1e4


def allowed_transitions(tag_list, scheme):
    """Boolean [K, K] matrix of legal tag→tag moves, plus legal first and last tags."""
    K = len(tag_list)
    allow = torch.ones(K, K, dtype=torch.bool)
    start = torch.ones(K, dtype=torch.bool)
    end = torch.ones(K, dtype=torch.bool)
    if scheme == "IO":
        return allow, start, end
    pre = ["O" if t == "O" else t[0] for t in tag_list]
    typ = [None if t == "O" else t[2:] for t in tag_list]
    for j in range(K):
        inside = pre[j] in ("I", "E")                          # tag j continues an open span
        start[j] = not inside
        if scheme == "BIOES":
            end[j] = pre[j] not in ("B", "I")
        for i in range(K):
            if scheme == "BIO":
                allow[i, j] = (not inside) or (pre[i] in ("B", "I") and typ[i] == typ[j])
            else:
                open_i = pre[i] in ("B", "I")
                allow[i, j] = (inside and open_i and typ[i] == typ[j]) or (not inside and not open_i)
    return allow, start, end


class CRF(nn.Module):
    def __init__(self, num_tags, allow=None, start_ok=None, end_ok=None):
        super().__init__()
        K = num_tags
        self.trans = nn.Parameter(torch.zeros(K, K))
        self.start = nn.Parameter(torch.zeros(K))
        self.end = nn.Parameter(torch.zeros(K))
        self.register_buffer("allow", torch.ones(K, K, dtype=torch.bool) if allow is None else allow)
        self.register_buffer("start_ok", torch.ones(K, dtype=torch.bool) if start_ok is None else start_ok)
        self.register_buffer("end_ok", torch.ones(K, dtype=torch.bool) if end_ok is None else end_ok)

    def potentials(self):
        return (self.trans.masked_fill(~self.allow, NEG), self.start.masked_fill(~self.start_ok, NEG),
                self.end.masked_fill(~self.end_ok, NEG))

    def log_partition(self, em, mask):                         # em [B,T,K], mask [B,T] left-aligned
        """Forward algorithm in exp-space: log Σᵢ exp(αᵢ + Aᵢⱼ) = m + log((exp(α − m) @ exp(A))ⱼ), with m = maxᵢ αᵢ.
        A matrix product replaces a [B, K, K] logsumexp at every step; illegal moves (−10⁴) become exact zeros."""
        trans, start, end = self.potentials()
        exp_trans = trans.exp()
        alpha = start + em[:, 0]
        for t in range(1, em.size(1)):
            m = alpha.max(dim=1, keepdim=True).values
            nxt = torch.log((alpha - m).exp() @ exp_trans + 1e-30) + m + em[:, t]
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        return torch.logsumexp(alpha + end, dim=1)

    def score(self, em, tags, mask):
        trans, start, end = self.potentials()
        tags = tags.clamp(min=0)
        s = start[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, em.size(1)):
            step = trans[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            s = s + step * mask[:, t].to(em.dtype)
        last = tags.gather(1, (mask.sum(1) - 1).unsqueeze(1)).squeeze(1)
        return s + end[last]

    def nll(self, em, tags, mask):
        return self.log_partition(em, mask) - self.score(em, tags, mask)

    def viterbi_score(self, em, mask):
        trans, start, end = self.potentials()
        delta = start + em[:, 0]
        for t in range(1, em.size(1)):
            nxt = (delta.unsqueeze(2) + trans).max(dim=1).values + em[:, t]
            delta = torch.where(mask[:, t].unsqueeze(1), nxt, delta)
        return (delta + end).max(dim=1).values

    def viterbi(self, em, mask):
        trans, start, end = self.potentials()
        B, T, K = em.shape
        delta, back = start + em[:, 0], []
        for t in range(1, T):
            best, idx = (delta.unsqueeze(2) + trans).max(dim=1)
            delta = torch.where(mask[:, t].unsqueeze(1), best + em[:, t], delta)
            back.append(idx)
        score, last = (delta + end).max(dim=1)
        back = torch.stack(back, 1).cpu() if back else None      # [B, T-1, K]
        lengths, last = mask.sum(1).tolist(), last.cpu().tolist()
        paths = []
        for b in range(B):
            path = [last[b]]
            for t in range(lengths[b] - 1, 0, -1):
                path.append(back[b, t - 1, path[-1]].item())
            paths.append(path[::-1])
        return paths, score

**Check:** log Z and Viterbi must equal brute force over every tag sequence (T = 3, K = 5), and constrained Viterbi must
never output an illegal sequence under BIO or BIOES.

In [ ]:
torch.manual_seed(0)
_crf = CRF(5)
with torch.no_grad():
    for _p in _crf.parameters():
        _p.normal_()
_em, _mask = torch.randn(2, 3, 5), torch.tensor([[1, 1, 1], [1, 1, 0]], dtype=torch.bool)
for _b in range(2):                                            # brute force over every tag sequence
    _L = int(_mask[_b].sum())
    _all = [(_crf.score(_em[_b:_b + 1], torch.tensor([list(p) + [0] * (3 - _L)]), _mask[_b:_b + 1]).item(), p)
            for p in itertools.product(range(5), repeat=_L)]
    assert abs(_crf.log_partition(_em[_b:_b + 1], _mask[_b:_b + 1]).item() - torch.logsumexp(torch.tensor([s for s, _ in _all]), 0).item()) < 1e-4
    assert tuple(_crf.viterbi(_em[_b:_b + 1], _mask[_b:_b + 1])[0][0]) == max(_all)[1]
    assert abs(_crf.viterbi_score(_em[_b:_b + 1], _mask[_b:_b + 1]).item() - max(_all)[0]) < 1e-4
for _scheme in ("BIO", "BIOES"):                                # constrained Viterbi never emits an illegal sequence
    _tags = vocab["tags"][_scheme]
    _c = CRF(len(_tags), *allowed_transitions(_tags, _scheme))
    _paths, _ = _c.viterbi(torch.randn(16, 12, len(_tags)) * 3, torch.ones(16, 12, dtype=torch.bool))
    for _p in _paths:
        _t = [_tags[i] for i in _p]
        assert spans_to_tags(12, tags_to_spans(_t, _scheme), _scheme) == _t, _t   # legal sequences round-trip unchanged
print("CRF checks pass: log Z and Viterbi match brute force; constrained paths are always legal")

**Recorded result:** `CRF checks pass: log Z and Viterbi match brute force; constrained paths are always legal`.

### Step 2: The input layer

Each token becomes its word vector (frozen or trainable, 300-d), optionally concatenated with a 100-d character-CNN
summary of its spelling (50-d character embeddings, 100 filters of width 3, max-pooled). Dropout follows.

In [ ]:
def masked(x, mask):
    return x * mask.unsqueeze(-1).to(x.dtype)


class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim=50, filters=100, width=3):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=PAD)
        self.conv = nn.Conv1d(char_dim, filters, width, padding=width // 2)

    def forward(self, chars):                                  # [B,T,C] -> [B,T,filters]
        B, T, C = chars.shape
        flat = chars.reshape(B * T, C)
        x = F.relu(self.conv(self.emb(flat).transpose(1, 2))).masked_fill((flat == PAD).unsqueeze(1), 0.0)
        return x.max(dim=2).values.view(B, T, -1)


class Embedder(nn.Module):
    def __init__(self, matrix, freeze, char_cnn, dropout):
        super().__init__()
        self.word = nn.Embedding(len(vocab["word2id"]), EMB_DIM, padding_idx=PAD)
        if matrix is not None:
            self.word.weight.data.copy_(torch.as_tensor(matrix))
        else:                                                  # random init is always trainable
            nn.init.normal_(self.word.weight, 0.0, RANDOM_STD)
            with torch.no_grad():
                self.word.weight[PAD].zero_()
        self.word.weight.requires_grad = matrix is None or not freeze
        self.char = CharCNN(len(vocab["char2id"])) if char_cnn else None
        self.out_dim = EMB_DIM + (100 if char_cnn else 0)
        self.drop = nn.Dropout(dropout)

    def forward(self, words, chars):
        x = self.word(words)
        if self.char is not None:
            x = torch.cat([x, self.char(chars)], dim=-1)
        return self.drop(x)

### Step 3: Nine context encoders

Each encoder takes `(x, mask)` and returns token states `[B, T, d]`. Matched sizes keep the comparison fair: 256 per RNN
direction, d_model 256, and about 256 convolution channels.

**R2, window MLP** (Collobert et al. 2011): a shared projection, then each token's state is concatenated with its two
neighbours. It answers: how much of the task is purely local evidence?

In [ ]:
class WindowEncoder(nn.Module):        # R2: no context beyond ±1 token (Collobert et al. 2011)
    def __init__(self, d_in, dropout, hidden=256):
        super().__init__()
        self.proj = nn.Linear(d_in, hidden)
        self.drop = nn.Dropout(dropout)
        self.out_dim = 3 * hidden

    def forward(self, x, mask):
        h = masked(F.relu(self.proj(x)), mask)
        left, right = F.pad(h, (0, 0, 1, 0))[:, :-1], F.pad(h, (0, 0, 0, 1))[:, 1:]
        return self.drop(torch.cat([left, h, right], dim=-1))

**R3, TextCNN** (Kim 2014): parallel convolutions of widths 1, 3 and 5 with "same" padding, so every position keeps its
own features (pooled features would lose position, so they never feed the slot head).

In [ ]:
class TextCNN(nn.Module):              # R3: widths 1/3/5 × 128 channels, 2 layers, position-aligned (Kim 2014)
    def __init__(self, d_in, dropout, widths=(1, 3, 5), channels=128, layers=2):
        super().__init__()
        self.layers, d = nn.ModuleList(), d_in
        for _ in range(layers):
            self.layers.append(nn.ModuleList(nn.Conv1d(d, channels, k, padding=k // 2) for k in widths))
            d = channels * len(widths)
        self.drop = nn.Dropout(dropout)
        self.out_dim = d

    def forward(self, x, mask):
        m = mask.unsqueeze(1).to(x.dtype)
        h = x.transpose(1, 2) * m
        for convs in self.layers:
            h = self.drop(F.relu(torch.cat([c(h) for c in convs], dim=1))) * m
        return h.transpose(1, 2)

**R4, ID-CNN** (Strubell et al. 2017): width-3 convolutions with dilations 1, 2 and 4, iterated 4 times with shared
weights and residual connections: a 57-token receptive field without recurrence.

In [ ]:
class IDCNN(nn.Module):                # R4: dilations 1/2/4, block iterated 4× with shared weights (Strubell et al. 2017)
    def __init__(self, d_in, dropout, channels=256, dilations=(1, 2, 4), iterations=4):
        super().__init__()
        self.proj = nn.Linear(d_in, channels)
        self.block = nn.ModuleList(nn.Conv1d(channels, channels, 3, padding=d, dilation=d) for d in dilations)
        self.iterations = iterations
        self.drop = nn.Dropout(dropout)
        self.out_dim = channels

    def forward(self, x, mask):
        m = mask.unsqueeze(1).to(x.dtype)
        h = self.proj(x).transpose(1, 2) * m
        for _ in range(self.iterations):
            res = h
            for conv in self.block:
                h = F.relu(conv(h)) * m
            h = self.drop(h + res)
        return h.transpose(1, 2)

**R5 / R6, BiGRU / BiLSTM:** two bidirectional layers over packed sequences, so padding never leaks into the backward pass.

In [ ]:
class BiRNN(nn.Module):                # R5 / R6: 2 × 256 per direction, packed sequences
    def __init__(self, d_in, dropout, cell="lstm", hidden=256, layers=2):
        super().__init__()
        rnn = nn.LSTM if cell == "lstm" else nn.GRU
        self.rnn = rnn(d_in, hidden, num_layers=layers, bidirectional=True, batch_first=True,
                       dropout=dropout if layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.out_dim = 2 * hidden

    def forward(self, x, mask):
        packed = nn.utils.rnn.pack_padded_sequence(x, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        return self.drop(out)

**R7, Transformer** (Gilbert's encoder): pre-LN blocks whose attention adds a learned per-head bias for each clipped
relative distance (±16), so the layer knows direction. **R9, CNN → attention** reuses it with a width-3 convolution as
its input layer and 2 layers.

In [ ]:
class RelPosSelfAttention(nn.Module):  # per-head bias for each clipped relative distance (from baseline_gilbert.ipynb)
    def __init__(self, d_model, n_heads, dropout, max_rel=16):
        super().__init__()
        self.h, self.dk, self.max_rel = n_heads, d_model // n_heads, max_rel
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.rel_bias = nn.Embedding(2 * max_rel + 1, n_heads)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        scores = (q @ k.transpose(-1, -2)) / math.sqrt(self.dk)
        pos = torch.arange(T, device=x.device)
        rel = (pos[None, :] - pos[:, None]).clamp(-self.max_rel, self.max_rel) + self.max_rel
        scores = (scores + self.rel_bias(rel).permute(2, 0, 1)).masked_fill(~mask[:, None, None, :], float("-inf"))
        return self.out((self.drop(scores.softmax(-1)) @ v).transpose(1, 2).reshape(B, T, -1))


class TransformerLayer(nn.Module):     # pre-LN block
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = RelPosSelfAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = x + self.drop(self.attn(self.ln1(x), mask))
        return x + self.drop(self.ff(self.ln2(x)))


class TransformerEncoder(nn.Module):   # R7: 3 × d256, 4 heads; R9 (conv_first, 2 layers): local conv then global attention
    def __init__(self, d_in, dropout, d_model=256, n_heads=4, d_ff=512, layers=3, conv_first=False):
        super().__init__()
        self.conv_first = conv_first
        self.inp = nn.Conv1d(d_in, d_model, 3, padding=1) if conv_first else nn.Linear(d_in, d_model)
        self.layers = nn.ModuleList(TransformerLayer(d_model, n_heads, d_ff, dropout) for _ in range(layers))
        self.norm = nn.LayerNorm(d_model)
        self.out_dim = d_model

    def forward(self, x, mask):
        h = F.relu(self.inp(masked(x, mask).transpose(1, 2))).transpose(1, 2) if self.conv_first else self.inp(x)
        h = masked(h, mask)
        for layer in self.layers:
            h = layer(h, mask)
        return masked(self.norm(h), mask)

**R8, CNN → BiGRU:** local n-gram composition first, then global context.

In [ ]:
class ConvBiGRU(nn.Module):            # R8: one width-3 conv, then a 1-layer BiGRU
    def __init__(self, d_in, dropout, channels=256, hidden=256):
        super().__init__()
        self.conv = nn.Conv1d(d_in, channels, 3, padding=1)
        self.rnn = BiRNN(channels, dropout, cell="gru", hidden=hidden, layers=1)
        self.drop = nn.Dropout(dropout)
        self.out_dim = 2 * hidden

    def forward(self, x, mask):
        h = masked(F.relu(self.conv(masked(x, mask).transpose(1, 2))).transpose(1, 2), mask)
        return self.rnn(self.drop(h), mask)

In [ ]:
ENCODERS = {
    "window": lambda d, p: WindowEncoder(d, p),
    "textcnn": lambda d, p: TextCNN(d, p),
    "idcnn": lambda d, p: IDCNN(d, p),
    "bigru": lambda d, p: BiRNN(d, p, cell="gru"),
    "bilstm": lambda d, p: BiRNN(d, p, cell="lstm"),
    "transformer": lambda d, p: TransformerEncoder(d, p),
    "cnn_bigru": lambda d, p: ConvBiGRU(d, p),
    "cnn_attn": lambda d, p: TransformerEncoder(d, p, layers=2, conv_first=True),
}

### Step 4: The joint heads and loss

`JointModel` = embedder → encoder → two heads trained together. The loss is
λ · CE(intent) + (1 − λ) · slot loss, with λ = 0.5 for per-token softmax and 0.7 when the slot loss is a CRF sequence
NLL (it sums over tokens, so it is about 7× larger).

In [ ]:
class JointModel(nn.Module):
    """One network, one loss: λ·CE(intent) + (1−λ)·slot loss (token CE, or CRF negative log-likelihood)."""

    def __init__(self, cfg, matrix):
        super().__init__()
        self.cfg, self.scheme = cfg, cfg["scheme"]
        tags = vocab["tags"][self.scheme]
        self.embed = Embedder(matrix, cfg["freeze"], cfg["char_cnn"], cfg["dropout"])
        self.encoder = ENCODERS[cfg["family"]](self.embed.out_dim, cfg["dropout"])
        d = self.encoder.out_dim
        self.drop = nn.Dropout(cfg["dropout"])
        self.intent_head = nn.Linear(d, len(vocab["intent2id"]))
        self.slot_head = nn.Linear(d, len(tags))
        self.crf = CRF(len(tags), *allowed_transitions(tags, self.scheme)) if cfg["decoder"] == "crf" else None

    def forward(self, batch):
        mask = batch["mask"]
        h = self.encoder(self.embed(batch["words"], batch["chars"]), mask)
        pooled = h.masked_fill(~mask.unsqueeze(-1), NEG).max(dim=1).values
        return self.intent_head(self.drop(pooled)), self.slot_head(self.drop(h))

    def loss(self, batch):
        il, em = self(batch)
        l_int = F.cross_entropy(il, batch["intents"], label_smoothing=self.cfg["label_smoothing"])
        if self.crf is None:
            l_slot = F.cross_entropy(em.reshape(-1, em.size(-1)), batch["tags"].reshape(-1), ignore_index=IGNORE)
        else:
            l_slot = self.crf.nll(em, batch["tags"], batch["mask"]).mean()
        lam = self.cfg["lam"]
        return lam * l_int + (1 - lam) * l_slot, l_int.detach(), l_slot.detach()

    @torch.no_grad()
    def predict(self, batch):
        il, em = self(batch)
        if self.crf is None:
            paths = [t[:n] for t, n in zip(em.argmax(-1).tolist(), batch["mask"].sum(1).tolist())]
        else:
            paths, _ = self.crf.viterbi(em, batch["mask"])
        return il.argmax(-1).tolist(), paths

### Step 5: The classical joint models

**R1, joint triangular CRF** (Jeong & Lee 2008). One log-linear model scores an intent and a tag sequence together:
s(I, Y, X) = θ_Iᵀφ(X) + Σₜ (E[t, yₜ] + A[yₜ₋₁, yₜ] + C[I, yₜ]). Sparse features come from train with count ≥ 2;
with a vector source, each token's static vector adds dense features. log Z sums over all 60 intents, each with
its own CRF partition, so both tasks share one normalised likelihood. Regularisation is an explicit L2 penalty.

**R0, joint HMM.** P(I) · P(y₁|I) · Πₜ P(yₜ|yₜ₋₁, I) · P(end|y_T, I) · Πₜ P(xₜ|yₜ, I), where slot-tag emissions
are shared across intents and only `O` emissions depend on the intent (a dense table would need 35.3M numbers).
Counts come from train with add-k smoothing; an UNK column is estimated from train singletons.

In [ ]:
def token_features(tokens, t):
    w = tokens[t]
    feats = [f"w={w}"]
    for off in (-2, -1, 1, 2):
        j = t + off
        feats.append(f"w{off:+d}=" + (tokens[j] if 0 <= j < len(tokens) else ("<s>" if j < 0 else "</s>")))
    for k in range(1, 5):
        if len(w) >= k:
            feats += [f"pre{k}={w[:k]}", f"suf{k}={w[-k:]}"]
    padded = f"<{w}>"
    feats += [f"c3={padded[i:i + 3]}" for i in range(len(padded) - 2)]
    return feats


def utterance_features(tokens):
    ext = ["<s>"] + tokens + ["</s>"]
    return [f"u={w}" for w in tokens] + [f"b={a}_{b}" for a, b in zip(ext, ext[1:])]


_FEATS = {}


def feature_dicts(min_count=2):
    if "dicts" not in _FEATS:
        tok_c, utt_c = collections.Counter(), collections.Counter()
        for ex in examples["train"]:                           # fitted on train only
            for t in range(len(ex["tokens"])):
                tok_c.update(token_features(ex["tokens"], t))
            utt_c.update(utterance_features(ex["tokens"]))
        tok = {f: i + 1 for i, f in enumerate(sorted(f for f, c in tok_c.items() if c >= min_count))}   # 0 = padding
        utt = {f: i + 1 for i, f in enumerate(sorted(f for f, c in utt_c.items() if c >= min_count))}
        _FEATS["dicts"] = (tok, utt)
    return _FEATS["dicts"]


def encoded_tricrf(split, scheme="BIO"):
    key = ("tricrf", split, scheme)
    if key not in _ENCODED:
        tok, utt = feature_dicts()
        out = []
        for ex, x in zip(examples[split], encoded(split, scheme)):
            feats = [[tok[f] for f in token_features(ex["tokens"], t) if f in tok] or [0] for t in range(len(ex["tokens"]))]
            out.append({**x, "tok_feats": feats, "utt_feats": [utt[f] for f in utterance_features(ex["tokens"]) if f in utt] or [0]})
        _ENCODED[key] = out
    return _ENCODED[key]

In [ ]:
class TriCRF(nn.Module):
    def __init__(self, cfg, matrix):
        super().__init__()
        self.cfg, self.scheme = cfg, cfg["scheme"]
        tok, utt = feature_dicts()
        tags = vocab["tags"][self.scheme]
        K, NI = len(tags), len(vocab["intent2id"])
        self.tok_w = nn.Embedding(len(tok) + 1, K, padding_idx=0)
        self.utt_w = nn.Embedding(len(utt) + 1, NI, padding_idx=0)
        nn.init.zeros_(self.tok_w.weight)
        nn.init.zeros_(self.utt_w.weight)
        self.intent_bias = nn.Parameter(torch.zeros(NI))
        self.compat = nn.Parameter(torch.zeros(NI, K))         # learned intent–tag compatibility C[I, y]
        self.crf = CRF(K, *allowed_transitions(tags, self.scheme))
        self.vec = None
        if matrix is not None:                                 # dense features from the static vectors
            self.vec = nn.Embedding.from_pretrained(torch.as_tensor(matrix), freeze=cfg["freeze"], padding_idx=PAD)
            self.vec_tok, self.vec_utt = nn.Linear(EMB_DIM, K), nn.Linear(EMB_DIM, NI)

    def scores(self, batch):
        em = self.tok_w(batch["tok_feats"]).sum(2)                 # [B,T,K]
        si = self.utt_w(batch["utt_feats"]).sum(1) + self.intent_bias   # [B,NI]
        if self.vec is not None:
            v, m = self.vec(batch["words"]), batch["mask"].unsqueeze(-1).to(em.dtype)
            em = em + self.vec_tok(v)
            si = si + self.vec_utt((v * m).sum(1) / m.sum(1))
        return si, em

    def _per_intent(self, em):                                 # [B,T,K] -> [B*NI,T,K]
        B, T, K = em.shape
        return (em.unsqueeze(1) + self.compat.view(1, -1, 1, K)).reshape(-1, T, K)

    def loss(self, batch):
        si, em = self.scores(batch)
        B, NI = si.shape
        logz = self.crf.log_partition(self._per_intent(em), batch["mask"].repeat_interleave(NI, 0)).view(B, NI)
        gold_i = batch["intents"]
        gold = si.gather(1, gold_i[:, None]).squeeze(1) + self.crf.score(em + self.compat[gold_i].unsqueeze(1), batch["tags"], batch["mask"])
        nll = (torch.logsumexp(si + logz, dim=1) - gold).mean()
        l2 = sum(p.pow(2).sum() for p in self.parameters() if p.requires_grad)
        return nll + self.cfg["weight_decay"] * l2, nll.detach(), nll.detach()

    @torch.no_grad()
    def predict(self, batch):
        si, em = self.scores(batch)
        B, NI = si.shape
        vit = self.crf.viterbi_score(self._per_intent(em), batch["mask"].repeat_interleave(NI, 0)).view(B, NI)
        best = (si + vit).argmax(1)                            # joint argmax over (intent, tag sequence)
        paths, _ = self.crf.viterbi(em + self.compat[best].unsqueeze(1), batch["mask"])
        return best.tolist(), paths

In [ ]:
class JointHMM:
    def __init__(self, k=0.1, scheme="BIO"):
        self.k, self.scheme = k, scheme

    def fit(self, data):
        NI, K, V = len(vocab["intent2id"]), len(vocab["tags"][self.scheme]), len(vocab["word2id"])
        allow, s_ok, e_ok = (a.numpy() for a in allowed_transitions(vocab["tags"][self.scheme], self.scheme))
        prior, start = np.zeros(NI), np.zeros((NI, K))
        trans = np.zeros((NI, K, K + 1))                        # last column = END
        emit_tag, emit_o = np.zeros((K, V)), np.zeros((NI, V))
        counts = vocab["word_counts"]
        singleton = np.zeros(V, dtype=bool)
        for w, i in vocab["word2id"].items():
            singleton[i] = counts.get(w, 0) == 1
        for x in data:
            i, w, y = x["intent"], x["words"], x["tags"]
            prior[i] += 1
            start[i, y[0]] += 1
            for a, b in zip(y, y[1:]):
                trans[i, a, b] += 1
            trans[i, y[-1], K] += 1
            for wt, yt in zip(w, y):
                if yt == 0:
                    emit_o[i, wt] += 1
                else:
                    emit_tag[yt, wt] += 1
        emit_tag[:, UNK] = (emit_tag * singleton).sum(1)          # UNK behaves like a train singleton
        emit_o[:, UNK] = (emit_o * singleton).sum(1)
        valid_t = np.concatenate([allow, e_ok[:, None]], axis=1)[None]

        def norm(c, valid):
            c = np.where(valid, c + self.k, 0.0)
            with np.errstate(divide="ignore"):
                return np.log(c / c.sum(-1, keepdims=True))

        word_valid = np.ones(V, dtype=bool)
        word_valid[PAD] = False
        self.log_prior = torch.tensor(np.log(prior / prior.sum()), dtype=torch.float32)
        self.log_start = torch.tensor(norm(start, s_ok[None]), dtype=torch.float32)
        self.log_trans = torch.tensor(norm(trans, valid_t), dtype=torch.float32)
        self.log_emit_tag = torch.tensor(norm(emit_tag, word_valid[None]), dtype=torch.float32)
        self.log_emit_o = torch.tensor(norm(emit_o, word_valid[None]), dtype=torch.float32)
        self.n_params = NI + NI * K + NI * K * (K + 1) + (K - 1) * (V - 1) + NI * (V - 1)
        return self

    @torch.no_grad()
    def predict(self, batch):
        words, mask = batch["words"], batch["mask"]
        B, T = words.shape
        NI, K = self.log_start.shape
        e = self.log_emit_tag[:, words].permute(1, 2, 0).unsqueeze(1).repeat(1, NI, 1, 1)   # [B,NI,T,K]
        e[..., 0] = self.log_emit_o[:, words].permute(1, 0, 2)                                # O emissions depend on I
        tr, end = self.log_trans[:, :, :K], self.log_trans[:, :, K]
        alpha = self.log_start.unsqueeze(0) + e[:, :, 0]
        for t in range(1, T):
            nxt = torch.logsumexp(alpha.unsqueeze(3) + tr.unsqueeze(0), dim=2) + e[:, :, t]
            alpha = torch.where(mask[:, t].view(B, 1, 1), nxt, alpha)
        intent = (self.log_prior + torch.logsumexp(alpha + end.unsqueeze(0), dim=2)).argmax(1)   # marginalise the tags
        eb, trb = e[torch.arange(B), intent], tr[intent]                                           # Viterbi under that intent
        delta, back = self.log_start[intent] + eb[:, 0], []
        for t in range(1, T):
            best, idx = (delta.unsqueeze(2) + trb).max(dim=1)
            delta = torch.where(mask[:, t].unsqueeze(1), best + eb[:, t], delta)
            back.append(idx)
        last = (delta + end[intent]).argmax(1).tolist()
        paths = []
        for b, n in enumerate(mask.sum(1).tolist()):
            path = [last[b]]
            for t in range(n - 1, 0, -1):
                path.append(back[t - 1][b, path[-1]].item())
            paths.append(path[::-1])
        return intent.tolist(), paths

### Step 6: The BiLSTM-based variants V1–V17

All 17 share the BiLSTM's embedder, 2 × 256 encoder and recipe, and change exactly one thing.

| ID | Variant | What changes | Reference |
|---|---|---|---|
| V1 / V2 / V3 | CRF with BIO / BIOES / IO | Constrained CRF decoder under three tag schemes (`decoder="crf"`, no new class) | Lafferty et al. 2001 |
| V4 | SlotRefine | Tag once, embed the predicted tags, tag again | Wu et al. 2020 |
| V5 | Boundary → type | Find span boundaries (B/I/O), then classify each span | Ma, Sun & Li 2024 |
| V6 | Biaffine spans | Score every span (≤ 10 tokens) for every type at once | Dozat & Manning 2017; Yu et al. 2020 |
| V7 | Semi-Markov CRF | A CRF over whole segments instead of tokens | Sarawagi & Cohen 2004 |
| V8 | Pointer decoder | Generate (start, end, type) triples left to right | Vinyals et al. 2015 |
| V9 | Partial sharing | Shared first layer, task-specific second layers | — |
| V10 | Slot-gate | Intent context gates the slot features | Goo et al. 2018 |
| V11 | Stack-propagation | Token-level intent votes feed the slot decoder | Qin et al. 2019 |
| V12 | SF-ID | Iterative intent ↔ slot reinforcement | E et al. 2019 |
| V13 | CM-Net | Slot and intent memories read by every token | Liu et al. 2019 |
| V14 | Co-interactive | Two-way cross-attention between intent and slot streams | Qin et al. 2021 |
| V15 | Capsules | Words route to tag capsules, tag capsules route to intent capsules | Zhang et al. 2019 |
| V16 | Graph attention | Tokens plus an intent node, connected by GAT layers | GL-GIN (single-intent), Veličković et al. 2018 |
| V17 | Intent–tag compatibility | A learned C[intent, tag] with joint normalisation (the neural TriCRF) | Jeong & Lee 2008 |

**The shared base:**

In [ ]:
def gold_spans_from_batch(batch, scheme="BIO"):
    tags = vocab["tags"][scheme]
    return [tags_to_spans([tags[t] for t in row[:n]], scheme)
            for row, n in zip(batch["tags"].tolist(), batch["mask"].sum(1).tolist())]


class VariantBase(nn.Module):
    """Embedder + 2 × 256 BiLSTM shared by every variant; subclasses add heads, losses and decoding."""
    returns_spans = False

    def __init__(self, cfg, matrix, layers=2):
        super().__init__()
        self.cfg, self.scheme = cfg, cfg["scheme"]
        self.tags = vocab["tags"][self.scheme]
        self.embed = Embedder(matrix, cfg["freeze"], cfg["char_cnn"], cfg["dropout"])
        self.encoder = BiRNN(self.embed.out_dim, cfg["dropout"], cell="lstm", layers=layers)
        self.d = self.encoder.out_dim
        self.drop = nn.Dropout(cfg["dropout"])
        self.n_int = len(vocab["intent2id"])
        self.crf = CRF(len(self.tags), *allowed_transitions(self.tags, self.scheme)) if cfg["decoder"] == "crf" else None

    def encode(self, batch):
        return self.encoder(self.embed(batch["words"], batch["chars"]), batch["mask"])

    @staticmethod
    def pool(h, mask):
        return h.masked_fill(~mask.unsqueeze(-1), NEG).max(dim=1).values

    def intent_loss(self, logits, batch):
        return F.cross_entropy(logits, batch["intents"], label_smoothing=self.cfg["label_smoothing"])

    @staticmethod
    def token_ce(em, batch):
        return F.cross_entropy(em.reshape(-1, em.size(-1)), batch["tags"].reshape(-1), ignore_index=IGNORE)

    def slot_loss(self, em, batch):                            # token softmax, or CRF when the arm turns it on
        return self.token_ce(em, batch) if self.crf is None else self.crf.nll(em, batch["tags"], batch["mask"]).mean()

    def decode(self, em, mask):
        if self.crf is None:
            return [t[:n] for t, n in zip(em.argmax(-1).tolist(), mask.sum(1).tolist())]
        return self.crf.viterbi(em, mask)[0]

    def combine(self, l_int, l_slot):
        lam = self.cfg["lam"]
        return lam * l_int + (1 - lam) * l_slot, l_int.detach(), l_slot.detach()

**V4, SlotRefine:** the second pass sees the first pass's tags (gold with probability 0.5 during training).

In [ ]:
class SlotRefine(VariantBase):
    """V4 (Wu et al. 2020): pass 1 tags every token; pass 2 sees the pass-1 tags (embedded) and re-tags.
    Training feeds pass 2 gold tags with probability 0.5, otherwise pass-1 predictions, so training matches inference."""

    def __init__(self, cfg, matrix):
        super().__init__(cfg, matrix)
        K = len(self.tags)
        self.intent_head = nn.Linear(self.d, self.n_int)
        self.first = nn.Linear(self.d, K)
        self.tag_emb = nn.Embedding(K + 1, 64)                  # index K marks padding
        self.refine = BiRNN(self.d + 64, cfg["dropout"], cell="lstm", hidden=128, layers=1)
        self.second = nn.Linear(self.refine.out_dim, K)

    def _passes(self, batch, gold_prob):
        mask = batch["mask"]
        h = self.encode(batch)
        il = self.intent_head(self.drop(self.pool(h, mask)))
        em1 = self.first(self.drop(h))
        tags_in = em1.argmax(-1).detach()
        if gold_prob > 0:
            use_gold = torch.rand(tags_in.shape, device=tags_in.device) < gold_prob
            tags_in = torch.where(use_gold, batch["tags"].clamp(min=0), tags_in)
        tags_in = tags_in.masked_fill(~mask, len(self.tags))
        em2 = self.second(self.drop(self.refine(torch.cat([h, self.tag_emb(tags_in)], dim=-1), mask)))
        return il, em1, em2

    def loss(self, batch):
        il, em1, em2 = self._passes(batch, 0.5 if self.training else 0.0)
        return self.combine(self.intent_loss(il, batch), self.token_ce(em1, batch) + self.token_ce(em2, batch))

    @torch.no_grad()
    def predict(self, batch):
        il, _, em2 = self._passes(batch, 0.0)
        return il.argmax(-1).tolist(), [t[:n] for t, n in zip(em2.argmax(-1).tolist(), batch["mask"].sum(1).tolist())]

**V5, boundary → type:** head 1 tags B/I/O boundaries; head 2 types each proposed span from [start; end; mean] plus intent context.

In [ ]:
class BoundaryType(VariantBase):
    """V5 (Ma, Sun & Li 2024): head 1 finds slot boundaries (O/B/I); head 2 types each span from
    [h_start; h_end; mean(h_span); soft intent context]. Both heads and the intent head share one loss."""
    returns_spans = True

    def __init__(self, cfg, matrix):
        super().__init__(cfg, matrix)
        self.types = vocab["slot_types"]
        self.type2id = {t: i for i, t in enumerate(self.types)}
        self.intent_head = nn.Linear(self.d, self.n_int)
        self.boundary = nn.Linear(self.d, 3)
        self.intent_ctx = nn.Linear(self.n_int, 64, bias=False)
        self.typer = nn.Sequential(nn.Linear(3 * self.d + 64, 256), nn.ReLU(), nn.Dropout(cfg["dropout"]),
                                   nn.Linear(256, len(self.types)))
        bmap = torch.zeros(len(self.tags), dtype=torch.long)          # BIO tag id -> 0 O, 1 B, 2 I
        for i, t in enumerate(self.tags):
            bmap[i] = 0 if t == "O" else (1 if t.startswith("B-") else 2)
        self.register_buffer("bmap", bmap)

    def _reps(self, h, spans, ctx):
        reps = [torch.cat([h[b, s], h[b, e - 1], h[b, s:e].mean(0), ctx[b]]) for b, sp in enumerate(spans) for _, s, e in sp]
        return torch.stack(reps) if reps else None

    def loss(self, batch):
        mask, tags = batch["mask"], batch["tags"]
        h = self.encode(batch)
        il = self.intent_head(self.drop(self.pool(h, mask)))
        bt = torch.where(tags >= 0, self.bmap[tags.clamp(min=0)], torch.full_like(tags, IGNORE))
        l_slot = F.cross_entropy(self.boundary(self.drop(h)).reshape(-1, 3), bt.reshape(-1), ignore_index=IGNORE)
        gold = gold_spans_from_batch(batch, self.scheme)
        reps = self._reps(h, gold, self.intent_ctx(il.softmax(-1)))
        if reps is not None:
            target = torch.tensor([self.type2id[t] for sp in gold for t, _, _ in sp], device=h.device)
            l_slot = l_slot + F.cross_entropy(self.typer(self.drop(reps)), target)
        return self.combine(self.intent_loss(il, batch), l_slot)

    @torch.no_grad()
    def predict(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        il = self.intent_head(self.pool(h, mask))
        paths = [t[:n] for t, n in zip(self.boundary(h).argmax(-1).tolist(), mask.sum(1).tolist())]
        spans = [tags_to_spans([("O", "B-x", "I-x")[t] for t in p]) for p in paths]
        reps = self._reps(h, spans, self.intent_ctx(il.softmax(-1)))
        typ = self.typer(reps).argmax(-1).tolist() if reps is not None else []
        out, k = [], 0
        for sp in spans:
            out.append([(self.types[typ[k + i]], s, e) for i, (_, s, e) in enumerate(sp)])
            k += len(sp)
        return il.argmax(-1).tolist(), out

**V6, biaffine spans:** a score for every (start, end, label); greedy non-overlapping decoding.

In [ ]:
class BiaffineSpans(VariantBase):
    """V6 (Dozat & Manning 2017; Yu et al. 2020): s_k(i, j) = u_iᵀ U_k v_j + w_kᵀ[u_i; v_j] + b_k for every span with
    j − i < 10 and every label k (55 types + none); cross-entropy over all candidates; greedy non-overlapping decoding."""
    returns_spans = True
    MAX_W = 10

    def __init__(self, cfg, matrix, ds=128):
        super().__init__(cfg, matrix)
        self.types = vocab["slot_types"]
        self.type2id = {t: i + 1 for i, t in enumerate(self.types)}   # 0 = no slot
        L = len(self.types) + 1
        self.intent_head = nn.Linear(self.d, self.n_int)
        self.start_mlp = nn.Sequential(nn.Linear(self.d, ds), nn.ReLU())
        self.end_mlp = nn.Sequential(nn.Linear(self.d, ds), nn.ReLU())
        self.U = nn.Parameter(torch.zeros(L, ds, ds))
        self.lin_u, self.lin_v = nn.Linear(ds, L), nn.Linear(ds, L, bias=False)

    def _scores(self, h, mask):
        u, v = self.start_mlp(self.drop(h)), self.end_mlp(self.drop(h))
        s = torch.einsum("bid,kde,bje->bijk", u, self.U, v) + self.lin_u(u).unsqueeze(2) + self.lin_v(v).unsqueeze(1)
        T = h.size(1)
        i, j = torch.arange(T, device=h.device).view(-1, 1), torch.arange(T, device=h.device).view(1, -1)
        valid = (j >= i) & (j - i < self.MAX_W)
        valid = valid.unsqueeze(0) & mask.unsqueeze(2) & mask.unsqueeze(1)   # [B,T,T]
        return s, valid

    def loss(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        il = self.intent_head(self.drop(self.pool(h, mask)))
        s, valid = self._scores(h, mask)
        target = torch.zeros(valid.shape, dtype=torch.long, device=h.device)
        for b, sp in enumerate(gold_spans_from_batch(batch, self.scheme)):
            for t, st, e in sp:
                if e - 1 - st < self.MAX_W:
                    target[b, st, e - 1] = self.type2id[t]
        l_slot = F.cross_entropy(s[valid], target[valid])
        return self.combine(self.intent_loss(il, batch), l_slot)

    @torch.no_grad()
    def predict(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        il = self.intent_head(self.pool(h, mask))
        s, valid = self._scores(h, mask)
        prob, lab = s.softmax(-1).max(-1)
        keep = valid & (lab > 0)
        out = []
        for b in range(h.size(0)):
            cand = sorted(((prob[b, i, j].item(), i, j, lab[b, i, j].item()) for i, j in keep[b].nonzero().tolist()), reverse=True)
            taken, spans = set(), []
            for _, i, j, k in cand:                            # highest-scoring spans first, no overlaps
                if not taken.intersection(range(i, j + 1)):
                    taken.update(range(i, j + 1))
                    spans.append((self.types[k - 1], i, j + 1))
            out.append(sorted(spans, key=lambda x: x[1]))
        return il.argmax(-1).tolist(), out

**V7, semi-Markov CRF:** segments of up to 10 tokens, `O` only as single tokens; the forward algorithm runs over segment ends.

In [ ]:
class SemiCRF(nn.Module):
    """Semi-Markov CRF over labelled segments (Sarawagi & Cohen 2004). phi[b, i, d-1, l] scores the segment that starts
    at i, has length d and label l (0 = O, which callers restrict to single tokens). Transitions link consecutive segments."""

    def __init__(self, n_labels, max_len):
        super().__init__()
        self.max_len = max_len
        self.trans = nn.Parameter(torch.zeros(n_labels, n_labels))
        self.start = nn.Parameter(torch.zeros(n_labels))
        self.end = nn.Parameter(torch.zeros(n_labels))

    def _forward(self, phi, use_max):
        B, T, D, L = phi.shape
        beta, alphas, best_len, best_prev = [self.start.expand(B, L)], [], [], [None]
        for j in range(T):
            cand = torch.stack([phi[:, j - d + 1, d - 1] + beta[j - d + 1] for d in range(1, min(D, j + 1) + 1)], dim=1)
            if use_max:
                a, bd = cand.max(dim=1)
                best_len.append(bd + 1)
            else:
                a = torch.logsumexp(cand, dim=1)
            alphas.append(a)
            nxt = a.unsqueeze(2) + self.trans                    # incoming score for a segment starting at j + 1
            if use_max:
                v, bl = nxt.max(dim=1)
                best_prev.append(bl)
                beta.append(v)
            else:
                beta.append(torch.logsumexp(nxt, dim=1))
        return alphas, best_len, best_prev

    def log_partition(self, phi, lengths):
        alphas, _, _ = self._forward(phi, use_max=False)
        last = torch.stack(alphas, dim=1)[torch.arange(phi.size(0)), lengths - 1]
        return torch.logsumexp(last + self.end, dim=1)

    def score(self, phi, segments):                            # segments[b] = [(start, length, label), ...] covering the sequence
        total = []
        for b, segs in enumerate(segments):
            s = self.start[segs[0][2]] + self.end[segs[-1][2]]
            s = s + sum(phi[b, i, d - 1, l] for i, d, l in segs)
            s = s + sum(self.trans[a[2], c[2]] for a, c in zip(segs, segs[1:]))
            total.append(s)
        return torch.stack(total)

    def viterbi(self, phi, lengths):
        alphas, best_len, best_prev = self._forward(phi, use_max=True)
        out = []
        for b, n in enumerate(lengths.tolist()):
            final = alphas[n - 1][b] + self.end
            lab, j, segs = final.argmax().item(), n - 1, []
            while j >= 0:
                d = best_len[j][b, lab].item()
                i = j - d + 1
                segs.append((i, d, lab))
                if i > 0:
                    lab = best_prev[i][b, lab].item()
                j = i - 1
            out.append(segs[::-1])
        return out


class SemiCRFSpans(VariantBase):
    """V7: a semi-Markov CRF on BiLSTM span representations tanh(W_s h_i + W_e h_j + W_m mean(h_i..j) + length embedding)."""
    returns_spans = True
    MAX_W = 10

    def __init__(self, cfg, matrix, dr=128):
        super().__init__(cfg, matrix)
        self.types = vocab["slot_types"]
        self.type2id = {t: i + 1 for i, t in enumerate(self.types)}
        self.intent_head = nn.Linear(self.d, self.n_int)
        self.ws, self.we, self.wm = nn.Linear(self.d, dr), nn.Linear(self.d, dr, bias=False), nn.Linear(self.d, dr, bias=False)
        self.len_emb = nn.Embedding(self.MAX_W, dr)
        self.out = nn.Linear(dr, len(self.types) + 1)
        self.semi = SemiCRF(len(self.types) + 1, self.MAX_W)
        o_long = torch.zeros(self.MAX_W, len(self.types) + 1, dtype=torch.bool)
        o_long[1:, 0] = True                                     # O segments are single tokens
        self.register_buffer("o_long", o_long)

    def _phi(self, h):
        B, T, _ = h.shape
        hs = self.drop(h)
        S, E = self.ws(hs), self.we(hs)
        csum = F.pad(hs.cumsum(1), (0, 0, 1, 0))
        idx = torch.arange(T, device=h.device)
        reps = []
        for d in range(1, self.MAX_W + 1):                       # spans running past the end are computed but never used
            mean = (csum[:, (idx + d).clamp(max=T)] - csum[:, idx]) / d
            reps.append(torch.tanh(S + E[:, (idx + d - 1).clamp(max=T - 1)] + self.wm(mean) + self.len_emb.weight[d - 1]))
        return self.out(torch.stack(reps, dim=2)).masked_fill(self.o_long, NEG)

    def _gold_segments(self, batch):
        segs_all = []
        for spans, n in zip(gold_spans_from_batch(batch, self.scheme), batch["mask"].sum(1).tolist()):
            starts = {s: (t, e) for t, s, e in spans}
            segs, t = [], 0
            while t < n:
                if t in starts and starts[t][1] - t <= self.MAX_W:
                    segs.append((t, starts[t][1] - t, self.type2id[starts[t][0]]))
                    t = starts[t][1]
                else:
                    segs.append((t, 1, 0))
                    t += 1
            segs_all.append(segs)
        return segs_all

    def loss(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        il = self.intent_head(self.drop(self.pool(h, mask)))
        phi = self._phi(h)
        nll = self.semi.log_partition(phi, mask.sum(1)) - self.semi.score(phi, self._gold_segments(batch))
        return self.combine(self.intent_loss(il, batch), nll.mean())

    @torch.no_grad()
    def predict(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        il = self.intent_head(self.pool(h, mask))
        segs = self.semi.viterbi(self._phi(h), mask.sum(1))
        return il.argmax(-1).tolist(), [[(self.types[l - 1], i, i + d) for i, d, l in s if l > 0] for s in segs]

**Check:** the semi-CRF's log Z and Viterbi must equal brute force over every segmentation.

In [ ]:
torch.manual_seed(0)
_semi = SemiCRF(3, 2)
with torch.no_grad():
    for _p in _semi.parameters():
        _p.normal_()
_phi = torch.randn(1, 4, 2, 3)
_phi[:, :, 1, 0] = NEG                                           # O only as single tokens, as in SemiCRFSpans


def _segmentations(n, max_len, n_labels):
    if n == 0:
        yield []
        return
    for d in range(1, min(max_len, n) + 1):
        for l in range(n_labels):
            if l == 0 and d > 1:
                continue
            for rest in _segmentations(n - d, max_len, n_labels):
                yield [(n - d, d, l)] + rest


for _n in (3, 4):
    _segs = [sorted(s) for s in _segmentations(_n, 2, 3)]
    _scores = torch.stack([_semi.score(_phi, [s])[0] for s in _segs])
    assert abs(_semi.log_partition(_phi[:, :_n], torch.tensor([_n])).item() - torch.logsumexp(_scores, 0).item()) < 1e-4
    assert _semi.viterbi(_phi[:, :_n], torch.tensor([_n]))[0] == _segs[_scores.argmax().item()]
print("semi-Markov CRF: log Z and Viterbi match brute force over every segmentation")

**Recorded result:** `semi-Markov CRF: log Z and Viterbi match brute force over every segmentation`.

**V8, pointer decoder:** an LSTM decoder points at each span's start and end, then its type, until STOP; teacher forcing in training.

In [ ]:
class PointerDecoder(VariantBase):
    """V8 (pointer networks, Vinyals et al. 2015): an LSTM decoder emits (start, end, type) triples left to right
    until STOP; starts are restricted to after the previous slot. Teacher forcing in training, greedy decoding."""
    returns_spans = True
    MAX_SLOTS, MAX_W = 8, 10

    def __init__(self, cfg, matrix, hd=256, da=128):
        super().__init__(cfg, matrix)
        self.types = vocab["slot_types"]
        self.type2id = {t: i for i, t in enumerate(self.types)}
        self.intent_head = nn.Linear(self.d, self.n_int)
        self.init = nn.Linear(self.d, 2 * hd)
        self.type_emb = nn.Embedding(len(self.types) + 1, 32)
        self.cell = nn.LSTMCell(2 * self.d + 32, hd)
        self.att_h, self.att_s, self.att_v = nn.Linear(self.d, da), nn.Linear(hd, da, bias=False), nn.Linear(da, 1, bias=False)
        self.stop = nn.Linear(hd, 1)
        self.end_h, self.end_s, self.end_a = nn.Linear(self.d, da), nn.Linear(hd, da, bias=False), nn.Linear(self.d, da, bias=False)
        self.end_v = nn.Linear(da, 1, bias=False)
        self.typer = nn.Sequential(nn.Linear(hd + 2 * self.d, 256), nn.ReLU(), nn.Linear(256, len(self.types)))

    def _start_logits(self, hs, s, mask, min_start):
        sc = self.att_v(torch.tanh(hs + self.att_s(s).unsqueeze(1))).squeeze(-1)
        pos = torch.arange(mask.size(1), device=mask.device).unsqueeze(0)
        return torch.cat([sc.masked_fill(~mask | (pos < min_start.unsqueeze(1)), NEG), self.stop(s)], dim=1)   # index T = STOP

    def _end_logits(self, H, he, s, start, mask):
        hst = H[torch.arange(H.size(0)), start]
        sc = self.end_v(torch.tanh(he + self.end_s(s).unsqueeze(1) + self.end_a(hst).unsqueeze(1))).squeeze(-1)
        pos = torch.arange(H.size(1), device=H.device).unsqueeze(0)
        ok = mask & (pos >= start.unsqueeze(1)) & (pos < start.unsqueeze(1) + self.MAX_W)
        return sc.masked_fill(~ok, NEG), hst

    def _init_state(self, H, mask):
        s, c = torch.tanh(self.init(self.pool(H, mask))).chunk(2, dim=-1)
        return s.contiguous(), c.contiguous(), torch.zeros(H.size(0), 2 * self.d + 32, device=H.device)

    def loss(self, batch):
        mask = batch["mask"]
        H = self.encode(batch)
        B, T, _ = H.shape
        il = self.intent_head(self.drop(self.pool(H, mask)))
        Hd = self.drop(H)
        hs, he = self.att_h(Hd), self.end_h(Hd)
        gold = [[sp for sp in g if sp[2] - sp[1] <= self.MAX_W] for g in gold_spans_from_batch(batch, self.scheme)]
        s, c, inp = self._init_state(H, mask)
        min_start = torch.zeros(B, dtype=torch.long, device=H.device)
        dev, total, count = H.device, 0.0, 0
        for k in range(max(len(g) for g in gold) + 1):
            s, c = self.cell(inp, (s, c))
            has = torch.tensor([k < len(g) for g in gold], device=dev)
            active = torch.tensor([k <= len(g) for g in gold], device=dev)      # emitting a slot, or the final STOP
            t_start = torch.tensor([g[k][1] if k < len(g) else T for g in gold], device=dev)
            t_end = torch.tensor([g[k][2] - 1 if k < len(g) else 0 for g in gold], device=dev)
            t_type = torch.tensor([self.type2id[g[k][0]] if k < len(g) else 0 for g in gold], device=dev)
            l = F.cross_entropy(self._start_logits(hs, s, mask, min_start), t_start, reduction="none")
            el, hst = self._end_logits(Hd, he, s, t_start.clamp(max=T - 1), mask)
            hen = Hd[torch.arange(B), t_end]
            tl = self.typer(torch.cat([s, hst, hen], dim=-1))
            l = l + has.float() * (F.cross_entropy(el, t_end, reduction="none") + F.cross_entropy(tl, t_type, reduction="none"))
            total = total + (l * active.float()).sum()
            count += int(active.sum())
            inp = torch.cat([hst, hen, self.type_emb(torch.where(has, t_type, torch.full_like(t_type, len(self.types))))], dim=-1)
            min_start = torch.where(has, t_end + 1, min_start)
        return self.combine(self.intent_loss(il, batch), total / count)

    @torch.no_grad()
    def predict(self, batch):
        mask = batch["mask"]
        H = self.encode(batch)
        B, T, _ = H.shape
        il = self.intent_head(self.pool(H, mask))
        hs, he = self.att_h(H), self.end_h(H)
        s, c, inp = self._init_state(H, mask)
        min_start = torch.zeros(B, dtype=torch.long, device=H.device)
        done = torch.zeros(B, dtype=torch.bool, device=H.device)
        out = [[] for _ in range(B)]
        for _ in range(self.MAX_SLOTS):
            s, c = self.cell(inp, (s, c))
            st = self._start_logits(hs, s, mask, min_start).argmax(-1)
            done = done | (st == T)
            st = st.clamp(max=T - 1)
            el, hst = self._end_logits(H, he, s, st, mask)
            en = el.argmax(-1)
            hen = H[torch.arange(B), en]
            ty = self.typer(torch.cat([s, hst, hen], dim=-1)).argmax(-1)
            for b in (~done).nonzero().flatten().tolist():
                out[b].append((self.types[ty[b].item()], st[b].item(), en[b].item() + 1))
            if done.all():
                break
            inp = torch.cat([hst, hen, self.type_emb(ty)], dim=-1)
            min_start = torch.where(done, min_start, en + 1)
        return il.argmax(-1).tolist(), out

**Interaction variants V9–V17** share one loss and predict routine; each defines `_heads`.

In [ ]:
def _variant_loss(self, batch):                                   # default for interaction variants: λ·intent + (1−λ)·slot
    il, em, l_int = self._heads(batch)
    return self.combine(self.intent_loss(il, batch) if l_int is None else l_int, self.slot_loss(em, batch))


@torch.no_grad()
def _variant_predict(self, batch):
    il, em, _ = self._heads(batch)
    return il.argmax(-1).tolist(), self.decode(em, batch["mask"])


class InteractionBase(VariantBase):
    loss, predict = _variant_loss, _variant_predict

**V9, partial sharing:**

In [ ]:
class PartialShare(InteractionBase):
    """V9: one shared BiLSTM layer, then a task-specific BiLSTM layer per task, trained by one joint loss."""

    def __init__(self, cfg, matrix):
        super().__init__(cfg, matrix, layers=1)
        p = cfg["dropout"]
        self.intent_rnn, self.slot_rnn = BiRNN(self.d, p, layers=1), BiRNN(self.d, p, layers=1)
        self.intent_head, self.slot_head = nn.Linear(self.d, self.n_int), nn.Linear(self.d, len(self.tags))

    def _heads(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        il = self.intent_head(self.drop(self.pool(self.intent_rnn(h, mask), mask)))
        return il, self.slot_head(self.drop(self.slot_rnn(h, mask))), None

**V10, slot-gate** (uses token-to-token slot attention and attention-pooled intent context):

In [ ]:
def _attention_contexts(h, mask, att_slot, att_int):
    e = (att_slot(h) @ h.transpose(1, 2)).masked_fill(~mask.unsqueeze(1), NEG)        # token-to-token slot attention
    c_slot = e.softmax(-1) @ h
    a = att_int(h).squeeze(-1).masked_fill(~mask, NEG).softmax(-1)                     # intent attention pooling
    return c_slot, (a.unsqueeze(-1) * h).sum(1)


class SlotGate(InteractionBase):
    """V10 (Goo et al. 2018): g_t = vᵀ tanh(c_slot(t) + W·c_int); slot features h_t + g_t · c_slot(t)."""

    def __init__(self, cfg, matrix):
        super().__init__(cfg, matrix)
        d = self.d
        self.att_slot, self.att_int = nn.Linear(d, d), nn.Linear(d, 1)
        self.W, self.v = nn.Linear(d, d, bias=False), nn.Linear(d, 1, bias=False)
        self.intent_head, self.slot_head = nn.Linear(2 * d, self.n_int), nn.Linear(d, len(self.tags))

    def _heads(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        c_slot, c_int = _attention_contexts(h, mask, self.att_slot, self.att_int)
        g = self.v(torch.tanh(c_slot + self.W(c_int).unsqueeze(1)))
        il = self.intent_head(self.drop(torch.cat([self.pool(h, mask), c_int], -1)))
        return il, self.slot_head(self.drop(h + g * c_slot)), None

**V11, stack-propagation:**

In [ ]:
class StackProp(InteractionBase):
    """V11 (Qin et al. 2019): every token predicts the intent (auxiliary CE against the utterance intent); the utterance
    intent is the soft vote over tokens; the slot decoder sees each token's intent distribution (embedded)."""

    def __init__(self, cfg, matrix, de=64):
        super().__init__(cfg, matrix)
        self.token_intent = nn.Linear(self.d, self.n_int)
        self.intent_emb = nn.Linear(self.n_int, de, bias=False)
        self.slot_head = nn.Linear(self.d + de, len(self.tags))

    def _heads(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        ti = self.token_intent(self.drop(h))
        il = ti.log_softmax(-1).masked_fill(~mask.unsqueeze(-1), 0.0).sum(1)
        tgt = batch["intents"].unsqueeze(1).expand_as(mask)
        l_int = F.cross_entropy(ti[mask], tgt[mask], label_smoothing=self.cfg["label_smoothing"])
        return il, self.slot_head(self.drop(torch.cat([h, self.intent_emb(ti.softmax(-1))], -1))), l_int

**V12, SF-ID:**

In [ ]:
class SFID(InteractionBase):
    """V12 (simplified SF-ID, E et al. 2019): an SF subnet gates slot contexts with the intent vector, an ID subnet attends
    over the gated slot vectors to reinforce the intent vector; 3 alternating rounds."""

    def __init__(self, cfg, matrix, rounds=3):
        super().__init__(cfg, matrix)
        d = self.d
        self.att_slot, self.att_int = nn.Linear(d, d), nn.Linear(d, 1)
        self.W_sf, self.v_sf, self.W_id = nn.Linear(d, d, bias=False), nn.Linear(d, 1, bias=False), nn.Linear(d, d, bias=False)
        self.rounds = rounds
        self.intent_head, self.slot_head = nn.Linear(2 * d, self.n_int), nn.Linear(2 * d, len(self.tags))

    def _heads(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        c_slot, c_int = _attention_contexts(h, mask, self.att_slot, self.att_int)
        r_int = c_int
        for _ in range(self.rounds):
            r_slot = self.v_sf(torch.tanh(c_slot + self.W_sf(r_int).unsqueeze(1))) * c_slot
            att = (r_slot @ self.W_id(r_int).unsqueeze(-1)).squeeze(-1).masked_fill(~mask, NEG).softmax(-1)
            r_int = c_int + (att.unsqueeze(-1) * r_slot).sum(1)
        il = self.intent_head(self.drop(torch.cat([self.pool(h, mask), r_int], -1)))
        return il, self.slot_head(self.drop(torch.cat([h, r_slot], -1))), None

**V13, CM-Net:**

In [ ]:
class CMNet(InteractionBase):
    """V13 (simplified CM-Net, Liu et al. 2019): learned slot and intent memories; in each of 2 collaborative blocks every
    token reads both memories and updates its state, h ← LN(h + ReLU(W[h; slot read; intent read]))."""

    def __init__(self, cfg, matrix, d_m=256, blocks=2):
        super().__init__(cfg, matrix)
        self.proj = nn.Linear(self.d, d_m)
        self.M_slot = nn.Parameter(torch.randn(len(self.tags), d_m) * 0.1)
        self.M_int = nn.Parameter(torch.randn(self.n_int, d_m) * 0.1)
        self.blocks = nn.ModuleList(nn.Linear(3 * d_m, d_m) for _ in range(blocks))
        self.norms = nn.ModuleList(nn.LayerNorm(d_m) for _ in range(blocks))
        self.intent_head, self.slot_head = nn.Linear(d_m, self.n_int), nn.Linear(d_m, len(self.tags))

    def _heads(self, batch):
        mask = batch["mask"]
        h = self.proj(self.encode(batch))
        for lin, ln in zip(self.blocks, self.norms):
            s = (h @ self.M_slot.T).softmax(-1) @ self.M_slot
            i = (h @ self.M_int.T).softmax(-1) @ self.M_int
            h = masked(ln(h + F.relu(lin(self.drop(torch.cat([h, s, i], -1))))), mask)
        return self.intent_head(self.drop(self.pool(h, mask))), self.slot_head(self.drop(h)), None

**V14, co-interactive attention:**

In [ ]:
class CoInteractive(InteractionBase):
    """V14 (Qin et al. 2021): label attention builds intent and slot streams from the classifier weights; each stream
    cross-attends to the other (residual + LayerNorm), then a ±1-window feed-forward refines each position; 1 layer, d 256."""

    def __init__(self, cfg, matrix, d_m=256, heads=4):
        super().__init__(cfg, matrix)
        self.proj = nn.Linear(self.d, d_m)
        self.intent_head, self.slot_head = nn.Linear(d_m, self.n_int), nn.Linear(d_m, len(self.tags))
        self.att_s, self.att_i = (nn.MultiheadAttention(d_m, heads, dropout=0.1, batch_first=True) for _ in range(2))
        self.ffn_s, self.ffn_i = (nn.Sequential(nn.Linear(3 * d_m, d_m), nn.ReLU(), nn.Linear(d_m, d_m)) for _ in range(2))
        self.ln = nn.ModuleList(nn.LayerNorm(d_m) for _ in range(4))

    @staticmethod
    def _window(x, mask):
        x = masked(x, mask)
        return torch.cat([F.pad(x, (0, 0, 1, 0))[:, :-1], x, F.pad(x, (0, 0, 0, 1))[:, 1:]], -1)

    def _heads(self, batch):
        mask = batch["mask"]
        H = masked(self.proj(self.encode(batch)), mask)
        WI, WS = self.intent_head.weight, self.slot_head.weight
        HI = H + (H @ WI.T).softmax(-1) @ WI                       # label attention (tied to the classifiers)
        HS = H + (H @ WS.T).softmax(-1) @ WS
        HS2 = self.ln[0](HS + self.att_s(HS, HI, HI, key_padding_mask=~mask, need_weights=False)[0])   # slots read intent
        HI2 = self.ln[1](HI + self.att_i(HI, HS, HS, key_padding_mask=~mask, need_weights=False)[0])   # intent reads slots
        HS3 = masked(self.ln[2](HS2 + self.ffn_s(self.drop(self._window(HS2, mask)))), mask)
        HI3 = masked(self.ln[3](HI2 + self.ffn_i(self.drop(self._window(HI2, mask)))), mask)
        return self.intent_head(self.drop(self.pool(HI3, mask))), self.slot_head(self.drop(HS3)), None

**V15, capsules.** The docstring records the two changes needed to make routing trainable here. Even so, V15 memorises
the slots but not the intents (section 9), so it is flagged ⚠ in every table.

In [ ]:
class CapsuleNLU(InteractionBase):
    """V15 (Zhang et al. 2019): word capsules route to tag capsules by agreement (3 iterations); a word's tag scores are its
    routing logits; tag capsules route to intent capsules, whose lengths score the intents.

    Two changes from the paper keep it trainable here. (1) The routed sum is scaled by n_out / n_in, so uniform coupling
    gives every output capsule unit total weight. Dynamic routing was designed for many children and few parents
    (1,152 → 10 in Sabour et al. 2017); here ~7 words feed 111 tag capsules, so each received ~0.06 of a word and the
    squash, quadratic near 0, left every capsule and its gradient at ~0 (measured: slot logits flat, 0.00 slot F1 on the
    64-utterance smoke test). (2) A LayerNorm on the encoder states puts the prediction vectors at unit scale.
    Slot capsules are 64-d (the paper uses the LSTM width; 16 and 32 learned intents more slowly)."""

    def __init__(self, cfg, matrix, dc=64, iters=3):
        super().__init__(cfg, matrix)
        self.norm = nn.LayerNorm(self.d)
        self.W_slot = nn.Parameter(torch.randn(len(self.tags), self.d, dc) / math.sqrt(self.d))
        self.W_int = nn.Parameter(torch.randn(self.n_int, dc, 16) / math.sqrt(dc))
        self.iters = iters
        self.scale = nn.Parameter(torch.tensor(5.0))

    @staticmethod
    def squash(s):
        n2 = (s * s).sum(-1, keepdim=True)
        return n2 / (1 + n2) * s / torch.sqrt(n2 + 1e-8)

    def _route(self, u, valid):                                 # u [B, n_in, n_out, dc]; valid [B, n_in]
        gain = u.size(2) / valid.sum(1).clamp(min=1).view(-1, 1, 1)          # n_out / n_in
        b = torch.zeros(u.shape[:3], device=u.device)
        for r in range(self.iters):
            v = self.squash(gain * ((b.softmax(-1) * valid.unsqueeze(-1)).unsqueeze(-1) * u).sum(1))
            if r < self.iters - 1:
                b = b + (u * v.unsqueeze(1)).sum(-1)
        return b, v

    def _heads(self, batch):
        mask = batch["mask"]
        h = self.drop(self.norm(self.encode(batch)))
        b_slot, v_slot = self._route(torch.einsum("btd,kdc->btkc", h, self.W_slot), mask.to(h.dtype))
        u_int = torch.einsum("bkc,ncd->bknd", v_slot, self.W_int)
        _, v_int = self._route(u_int, torch.ones(u_int.shape[:2], device=h.device))
        return self.scale * v_int.norm(dim=-1), b_slot, None

**V16, graph attention:**

In [ ]:
class GATLayer(nn.Module):
    def __init__(self, d, heads=4, dropout=0.1):
        super().__init__()
        self.h, self.dk = heads, d // heads
        self.W = nn.Linear(d, d, bias=False)
        self.a_src, self.a_dst = nn.Parameter(torch.randn(heads, self.dk) * 0.1), nn.Parameter(torch.randn(heads, self.dk) * 0.1)
        self.drop, self.ln = nn.Dropout(dropout), nn.LayerNorm(d)

    def forward(self, x, adj):                                  # x [B,N,d], adj [B,N,N] bool
        B, N, _ = x.shape
        z = self.W(x).view(B, N, self.h, self.dk)
        e = F.leaky_relu((z * self.a_src).sum(-1).unsqueeze(2) + (z * self.a_dst).sum(-1).unsqueeze(1), 0.2)
        att = self.drop(e.masked_fill(~adj.unsqueeze(-1), NEG).softmax(dim=2))
        return self.ln(x + F.elu(torch.einsum("bijh,bjhd->bihd", att, z).reshape(B, N, -1)))


class GraphInteraction(InteractionBase):
    """V16 (single-intent adaptation of GL-GIN; GAT layers): nodes are the tokens plus one intent node; edges link tokens
    within ±1 and every token with the intent node; 2 graph-attention layers."""

    def __init__(self, cfg, matrix, d_m=256, layers=2):
        super().__init__(cfg, matrix)
        self.proj = nn.Linear(self.d, d_m)
        self.gat = nn.ModuleList(GATLayer(d_m) for _ in range(layers))
        self.intent_head, self.slot_head = nn.Linear(2 * d_m, self.n_int), nn.Linear(d_m, len(self.tags))

    def _heads(self, batch):
        mask = batch["mask"]
        h = masked(self.proj(self.encode(batch)), mask)
        B, T, _ = h.shape
        x = torch.cat([self.pool(h, mask).unsqueeze(1), h], 1)            # node 0 is the intent node
        valid = torch.cat([torch.ones(B, 1, dtype=torch.bool, device=h.device), mask], 1)
        idx = torch.arange(T + 1, device=h.device)
        local = (idx.view(-1, 1) - idx.view(1, -1)).abs() <= 1
        local[0, :] = True
        local[:, 0] = True
        adj = local.unsqueeze(0) & valid.unsqueeze(1) & valid.unsqueeze(2)
        for layer in self.gat:
            x = layer(x, adj) * valid.unsqueeze(-1).to(x.dtype)
        tok = x[:, 1:]
        il = self.intent_head(self.drop(torch.cat([x[:, 0], self.pool(tok, mask)], -1)))
        return il, self.slot_head(self.drop(tok)), None

**V17, intent–tag compatibility** (joint normalisation over 60 intents, joint argmax decoding):

In [ ]:
class CompatJoint(VariantBase):
    """V17: BiLSTM intent scores and emissions plus a learned intent–tag compatibility C[I, y], normalised jointly over
    (intent, tag sequence) as in the triangular CRF, with joint argmax decoding (the neural counterpart of R1)."""

    def __init__(self, cfg, matrix):
        super().__init__({**cfg, "decoder": "crf"}, matrix)
        self.intent_head, self.slot_head = nn.Linear(self.d, self.n_int), nn.Linear(self.d, len(self.tags))
        self.compat = nn.Parameter(torch.zeros(self.n_int, len(self.tags)))

    def _scores(self, batch):
        mask = batch["mask"]
        h = self.encode(batch)
        return self.intent_head(self.drop(self.pool(h, mask))), self.slot_head(self.drop(h))

    def _per_intent(self, em):
        B, T, K = em.shape
        return (em.unsqueeze(1) + self.compat.view(1, -1, 1, K)).reshape(-1, T, K)

    def loss(self, batch):
        si, em = self._scores(batch)
        B, NI = si.shape
        logz = self.crf.log_partition(self._per_intent(em), batch["mask"].repeat_interleave(NI, 0)).view(B, NI)
        gi = batch["intents"]
        gold = si.gather(1, gi[:, None]).squeeze(1) + self.crf.score(em + self.compat[gi].unsqueeze(1), batch["tags"], batch["mask"])
        nll = (torch.logsumexp(si + logz, 1) - gold).mean()
        return nll, nll.detach(), nll.detach()

    @torch.no_grad()
    def predict(self, batch):
        si, em = self._scores(batch)
        B, NI = si.shape
        vit = self.crf.viterbi_score(self._per_intent(em), batch["mask"].repeat_interleave(NI, 0)).view(B, NI)
        best = (si + vit).argmax(1)
        return best.tolist(), self.crf.viterbi(em + self.compat[best].unsqueeze(1), batch["mask"])[0]

**The variant registry** (`VARIANT_SPECS` maps each V-id to its overrides on the BiLSTM configuration):

In [ ]:
VARIANTS = {"slotrefine": SlotRefine, "boundary_type": BoundaryType, "biaffine": BiaffineSpans, "semicrf": SemiCRFSpans,
            "pointer": PointerDecoder, "partial_share": PartialShare, "slot_gate": SlotGate, "stack_prop": StackProp,
            "sfid": SFID, "cmnet": CMNet, "cointeractive": CoInteractive, "capsule": CapsuleNLU, "graph": GraphInteraction,
            "compat": CompatJoint}
VARIANT_SPECS = {   # V-id -> overrides on the BiLSTM family config (λ 0.7 for sequence-level slot losses)
    "V1": dict(decoder="crf", scheme="BIO", lam=0.7), "V2": dict(decoder="crf", scheme="BIOES", lam=0.7),
    "V3": dict(decoder="crf", scheme="IO", lam=0.7), "V4": dict(variant="slotrefine"), "V5": dict(variant="boundary_type"),
    "V6": dict(variant="biaffine"), "V7": dict(variant="semicrf", lam=0.7), "V8": dict(variant="pointer"),
    "V9": dict(variant="partial_share"), "V10": dict(variant="slot_gate"), "V11": dict(variant="stack_prop"),
    "V12": dict(variant="sfid"), "V13": dict(variant="cmnet"), "V14": dict(variant="cointeractive"),
    "V15": dict(variant="capsule"), "V16": dict(variant="graph"), "V17": dict(variant="compat"),
}

## 7. Training and evaluation

### Step 7: One configuration dict per run

`BASE` lists every switch; `FAMILY_DEFAULTS` adjusts it per family (the Transformer uses warmup and clipping at 1; the
TriCRF has no dropout and uses an explicit L2 penalty). `family_cfg(family, **overrides)` builds a run's configuration.

In [ ]:
BASE = dict(family="bilstm", emb="glove6b", freeze=True, emb_lr_mult=0.1, char_cnn=False, word_dropout=0.0,
            decoder="softmax", scheme="BIO", lam=0.5, label_smoothing=0.0, lr=1e-3, dropout=0.3, weight_decay=0.01,
            schedule="plateau", warmup_epochs=0, clip=5.0, batch_size=32, max_epochs=80, patience=10, device="mps")
FAMILY_DEFAULTS = {
    "transformer": dict(lr=5e-4, dropout=0.2, schedule="warmup_linear", warmup_epochs=1, clip=1.0),
    "tricrf": dict(decoder="joint", lr=3e-3, dropout=0.0, weight_decay=1e-5),
}
REPRESENTATIVES = ["tricrf", "window", "textcnn", "idcnn", "bigru", "bilstm", "transformer", "cnn_bigru", "cnn_attn"]


def family_cfg(family, **over):
    return {**BASE, **FAMILY_DEFAULTS.get(family, {}), "family": family, **over}

### Step 8: Build a model and count its parameters

`load_matrix` rescales a pretrained table to std 0.4. `count_system_params` counts every unique tensor of the system,
frozen word vectors included, plus floating-point buffers (boolean masks are structure, not weights).

In [ ]:
def load_matrix(name):
    if name == "random":
        return None
    matrix, info = embedding_matrix(name)
    return (matrix * (RANDOM_STD / info["table_std"])).astype(np.float32)


def make_model(cfg):
    matrix = load_matrix(cfg["emb"])
    if cfg.get("variant"):                                      # BiLSTM-based variants V4–V17 (defined below)
        return VARIANTS[cfg["variant"]](cfg, matrix)
    return TriCRF(cfg, matrix) if cfg["family"] == "tricrf" else JointModel(cfg, matrix)


def count_system_params(*models):
    """Unique tensors of every model, frozen ones included, plus floating-point buffers (boolean masks are structure)."""
    seen, n = set(), 0
    for m in models:
        if isinstance(m, JointHMM):
            n += m.n_params
            continue
        for t in list(m.parameters()) + [b for b in m.buffers() if b.is_floating_point()]:
            if id(t) not in seen:
                seen.add(id(t))
                n += t.numel()
    return n


def word_table_params(model):
    return sum(m.weight.numel() for m in model.modules()
               if isinstance(m, nn.Embedding) and m.num_embeddings == len(vocab["word2id"]) and m.embedding_dim == EMB_DIM)

### Step 9: Optimiser, schedule and word dropout

AdamW with three parameter groups (decayed weights, undecayed biases and norms, and the word table with its own learning
rate). The schedule is reduce-on-plateau (halve after 4 epochs without improvement) or, for the Transformer, warmup then
linear decay. Word dropout replaces token w by `<UNK>` with probability α / (α + count(w)); its characters stay visible.

In [ ]:
def make_optimizer(model, cfg, steps_per_epoch):
    emb_ids = {id(p) for n, p in model.named_parameters() if n in ("embed.word.weight", "vec.weight")}
    decay, no_decay, emb = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if id(p) in emb_ids:
            emb.append(p)
        elif p.ndim < 2 or "ln" in n or "norm" in n:
            no_decay.append(p)
        else:
            decay.append(p)
    wd = 0.0 if cfg["family"] == "tricrf" else cfg["weight_decay"]      # the TriCRF uses an explicit L2 term instead
    groups = [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0},
              {"params": emb, "weight_decay": 0.0, "lr": cfg["lr"] * (1.0 if cfg["emb"] == "random" else cfg["emb_lr_mult"])}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], lr=cfg["lr"])
    if cfg["schedule"] == "warmup_linear":
        warm, total = cfg["warmup_epochs"] * steps_per_epoch, cfg["max_epochs"] * steps_per_epoch
        lam = lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm))
        return opt, torch.optim.lr_scheduler.LambdaLR(opt, lam), "step"
    return opt, torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4), "plateau"


_DROP = {}


def drop_probs(alpha):
    """Word dropout: replace word w by UNK with probability α / (α + count(w)); characters stay visible."""
    if alpha not in _DROP:
        p = torch.zeros(len(vocab["word2id"]))
        for w, i in vocab["word2id"].items():
            c = vocab["word_counts"].get(w, 0)
            if c:
                p[i] = alpha / (alpha + c)
        _DROP[alpha] = p
    return _DROP[alpha]

### Step 10: Evaluation

Dev loss plus the shared metric, and the predictions themselves (intent names and spans), which later analyses reuse.

In [ ]:
def split_data(cfg, split):
    return encoded_tricrf(split, cfg["scheme"]) if cfg["family"] == "tricrf" else encoded(split, cfg["scheme"])


def evaluate(model, cfg, split="dev", device=torch.device("cpu"), data=None, exs=None):
    """Loss and shared-metric scores on one split; also returns the predictions (intent names, spans)."""
    data = split_data(cfg, split) if data is None else data
    exs = examples[split] if exs is None else exs
    is_hmm = isinstance(model, JointHMM)
    if not is_hmm:
        model.eval()
    pred_i, pred_t = [None] * len(data), [None] * len(data)
    total, nb = 0.0, 0
    with torch.no_grad():
        for idx in batch_indices(data, 16 if is_hmm else 32, shuffle=False):
            batch = to_device(collate([data[i] for i in idx]), device)
            if not is_hmm:
                total += model.loss(batch)[0].item()
                nb += 1
            intents, paths = model.predict(batch)
            for k, i in enumerate(idx):
                pred_i[i], pred_t[i] = intents[k], paths[k]
    tags = vocab["tags"][cfg["scheme"]]
    spans = pred_t if getattr(model, "returns_spans", False) else [tags_to_spans([tags[t] for t in p], cfg["scheme"]) for p in pred_t]
    names = [ID2INTENT[i] for i in pred_i]
    res = score_predictions(exs, names, spans)
    res["loss"] = total / nb if nb else float("nan")
    return res, (names, spans)

### Step 11: The training loop

Per epoch it logs train loss (total, intent, slot), dev loss, dev intent accuracy, slot F1, exact match, the combined
score, the learning rate and elapsed time. It keeps the best dev-combined epoch, stops after 10 epochs without improvement
(at most 80 epochs), and returns the best epoch's dev predictions.

In [ ]:
def train_model(cfg, seed, train_data=None, eval_data=None, eval_exs=None, verbose=False):
    set_seed(seed)
    device = pick_device(cfg["device"])
    model = make_model(cfg).to(device)
    train_data = split_data(cfg, "train") if train_data is None else train_data
    opt, sched, mode = make_optimizer(model, cfg, math.ceil(len(train_data) / cfg["batch_size"]))
    p_drop = drop_probs(cfg["word_dropout"]).to(device) if cfg["word_dropout"] > 0 else None
    best, best_epoch, best_dev, best_state, best_pred, bad, history, t0 = -1.0, 0, None, None, None, 0, [], time.time()
    for epoch in range(1, cfg["max_epochs"] + 1):
        model.train()
        sums, n = torch.zeros(3, device=device), 0
        for idx in batch_indices(train_data, cfg["batch_size"], shuffle=True, seed=seed * 1000 + epoch):
            batch = to_device(collate([train_data[i] for i in idx]), device)
            if p_drop is not None:
                hide = (torch.rand(batch["words"].shape, device=device) < p_drop[batch["words"]]) & batch["mask"]
                batch["words"] = batch["words"].masked_fill(hide, UNK)
            loss, l_int, l_slot = model.loss(batch)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])
            opt.step()
            if mode == "step":
                sched.step()
            sums += torch.stack([loss.detach(), l_int, l_slot])
            n += 1
        dev, pred = evaluate(model, cfg, "dev", device, data=eval_data, exs=eval_exs)
        tr = (sums / n).tolist()
        history.append({"epoch": epoch, "train_loss": tr[0], "train_intent_loss": tr[1], "train_slot_loss": tr[2],
                        "dev_loss": dev["loss"], "dev_intent_acc": dev["intent_acc"], "dev_slot_f1": dev["slot_f1"],
                        "dev_exact_match": dev["exact_match"], "dev_combined": dev["combined"],
                        "lr": opt.param_groups[0]["lr"], "elapsed": time.time() - t0})
        if verbose:
            h = history[-1]
            print(f"epoch {epoch:3d} | train {h['train_loss']:.4f} | dev loss {h['dev_loss']:.4f} | intent {h['dev_intent_acc']:.2f} "
                  f"| slot F1 {h['dev_slot_f1']:.2f} | combined {h['dev_combined']:.2f} | {h['elapsed']:.0f}s", flush=True)
        if dev["combined"] > best:
            best, best_epoch, best_dev, best_pred, bad = dev["combined"], epoch, dev, pred, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if mode == "plateau":
            sched.step(dev["combined"])
        if bad >= cfg["patience"]:
            break
    model.load_state_dict(best_state)
    return model, {"best_epoch": best_epoch, "dev": best_dev, "history": history, "seconds": time.time() - t0,
                   "dev_pred": {"intents": best_pred[0], "spans": [[list(s) for s in sp] for sp in best_pred[1]]}}

### Step 12: Cached runs

`run_experiment(cfg, seed)` names each run by a hash of its configuration (the device excluded, since it changes speed,
not the experiment), the seed, `VOCAB_HASH` and `CODE_VERSION`. A finished run is a `.pt` checkpoint plus a `.json`
summary (configuration, dev scores, per-epoch history, dev predictions, parameter counts, timing), written atomically.

In [ ]:
def config_key(cfg, seed):
    ident = {k: v for k, v in cfg.items() if k != "device"}   # the device changes speed, not the experiment
    blob = json.dumps({**ident, "_seed": seed, "_vocab": VOCAB_HASH, "_code": CODE_VERSION}, sort_keys=True)
    return f"{cfg['family']}_{hashlib.md5(blob.encode()).hexdigest()[:12]}_s{seed}"


def run_experiment(cfg, seed, verbose=False):
    """Train (or reload from the run cache) one configuration with one seed; returns its summary."""
    key = config_key(cfg, seed)
    log_path = os.path.join(RUN_CACHE, key + ".json")
    if os.path.exists(log_path):
        with open(log_path) as f:
            return json.load(f)
    model, out = train_model(cfg, seed, verbose=verbose)
    torch.save(model.state_dict(), os.path.join(RUN_CACHE, key + ".pt"))
    summary = {"key": key, "config": cfg, "seed": seed, **out, "params": count_system_params(model),
               "word_table_params": word_table_params(model), "sec_per_epoch": out["seconds"] / len(out["history"])}
    with open(log_path + ".tmp", "w") as f:
        json.dump(summary, f)
    os.replace(log_path + ".tmp", log_path)                    # a log file means the run finished
    return summary


def run_hmm(k, scheme="BIO"):
    key = f"hmm_k{k}_{scheme}_{VOCAB_HASH}_{CODE_VERSION}"
    path = os.path.join(RUN_CACHE, key + ".json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    t0 = time.time()
    hmm = JointHMM(k, scheme).fit(encoded("train", scheme))
    dev, _ = evaluate(hmm, {"scheme": scheme}, "dev")
    summary = {"key": key, "config": {"family": "hmm", "k": k, "scheme": scheme}, "dev": dev,
               "params": hmm.n_params, "seconds": time.time() - t0}
    with open(path, "w") as f:
        json.dump(summary, f)
    return summary

## 8. Experiment registry and runner

**P0 grids.** Each family gets exactly 6 recipes, 1 seed each, on frozen GloVe 6B. Everything else is held at the
setting later phases start from: the same data, vocabulary and batch size (32), per-token softmax (no CRF), and λ = 0.5.

| Family | Recipe grid |
|---|---|
| Window, TextCNN, ID-CNN, BiGRU, BiLSTM, CNN→BiGRU, CNN→attention | lr ∈ {5e-4, 1e-3, 2e-3} × dropout ∈ {0.2, 0.4} |
| Transformer | lr ∈ {2.5e-4, 5e-4, 1e-3} × dropout ∈ {0.1, 0.2}, with warmup |
| TriCRF | lr ∈ {1e-3, 3e-3, 1e-2} × weight decay ∈ {1e-5, 1e-4} |
| HMM | add-k smoothing k ∈ {0.01, 0.1, 0.5, 1} (deterministic, seconds each) |

Why these choices: only the optimisation settings are architecture-specific, so only they are tuned; every factor that a
later phase tests (vectors, CRF, techniques, λ) stays off or neutral, so its effect does not leak into the recipe. Frozen
GloVe 6B matches P1a's conditions and Gilbert's baseline. An equal budget per family keeps the comparison fair
(Melis et al. 2018).

**Selection rule (fixed before training):** each family keeps its best dev-combined recipe; recipes within 0.3 points
count as tied, and the one nearest the family's standard default wins (lr 1e-3 / dropout 0.3; Transformer 5e-4 / 0.2;
TriCRF 3e-3 / 1e-5); remaining ties go to the higher score. Distances are rounded, so dropout 0.2 and 0.4 tie exactly
around 0.3 (a floating-point bug that favoured 0.2 was found and fixed on 5 November 2026).

In [ ]:
P0_GRIDS = {
    "default": [dict(lr=lr, dropout=p) for lr in (5e-4, 1e-3, 2e-3) for p in (0.2, 0.4)],
    "transformer": [dict(lr=lr, dropout=p) for lr in (2.5e-4, 5e-4, 1e-3) for p in (0.1, 0.2)],
    "tricrf": [dict(lr=lr, weight_decay=wd) for lr in (1e-3, 3e-3, 1e-2) for wd in (1e-5, 1e-4)],
}
P0_DEFAULTS = {"default": dict(lr=1e-3, dropout=0.3), "transformer": dict(lr=5e-4, dropout=0.2), "tricrf": dict(lr=3e-3, weight_decay=1e-5)}
HMM_K = [0.01, 0.1, 0.5, 1.0]
DEVICES_PATH, RECIPES_PATH = "configs/devices.json", "configs/recipes.json"


def _load_json(path, default):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default


DEVICE_OVERRIDES = {"tricrf": "mps"}   # under parallel load: 19.0 s/epoch on MPS vs 35.8 on CPU (section 9)


def device_for(family):
    return DEVICE_OVERRIDES.get(family) or _load_json(DEVICES_PATH, {}).get("plan", {}).get(family, "cpu")


def p0_jobs():
    return [(family_cfg(fam, device=device_for(fam), **g), 0)
            for fam in REPRESENTATIVES for g in P0_GRIDS.get(fam, P0_GRIDS["default"])]

In [ ]:
def select_recipe(family, results):
    """results: list of (recipe dict, dev combined). Best score; ties within 0.3 go to the recipe nearest the default."""
    default = P0_DEFAULTS.get(family, P0_DEFAULTS["default"])
    best = max(s for _, s in results)

    def distance(r):                                           # rounded, so |0.2 − 0.3| and |0.4 − 0.3| tie exactly
        return round(sum(abs(math.log10(r[k] / default[k])) if k in ("lr", "weight_decay") else abs(r[k] - default[k])
                         for k in r), 9)

    near = [(distance(r), -s, r) for r, s in results if s >= best - 0.3]
    return min(near, key=lambda t: (t[0], t[1]))[2]


def recipe(family):
    return _load_json(RECIPES_PATH, {}).get(family, {})


def rep_cfg(family, **over):
    """A family representative with its P0 recipe and device."""
    return family_cfg(family, device=device_for(family), **{**recipe(family), **over})


def cached_summary(cfg, seed):
    path = os.path.join(RUN_CACHE, config_key(cfg, seed) + ".json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

**Later phases** reuse the recipe: `rep_cfg(family, **overrides)` builds a family representative with its P0 recipe and
device. P1b uses the BiLSTM's best P1a source plus `random`.

In [ ]:
def p1a_jobs():
    return [(rep_cfg(fam, emb=emb), seed) for fam in REPRESENTATIVES for emb in EMBEDDING_NAMES for seed in SEEDS]


def best_embedding(family):
    """The embedding with the best 3-seed mean dev combined score for a family in P1a (all runs must be finished)."""
    means = {}
    for emb in EMBEDDING_NAMES:
        runs = [cached_summary(rep_cfg(family, emb=emb), s) for s in SEEDS]
        assert all(runs), f"P1a is not finished for {family} / {emb}"
        means[emb] = np.mean([r["dev"]["combined"] for r in runs])
    return max(means, key=means.get)


def p1b_jobs():
    embs = sorted({best_embedding("bilstm"), "random"})
    return [(rep_cfg("bilstm", emb=emb, **spec), seed) for spec in VARIANT_SPECS.values() for emb in embs for seed in SEEDS]


PHASES = {"P0": p0_jobs, "P1a": p1a_jobs, "P1b": p1b_jobs}


def phase_jobs(phase):
    return PHASES[phase]()

**Helpers** for reloading a cached run's model and timing training steps (used by the speed benchmark):

In [ ]:
def load_run_model(summary, device="cpu"):
    model = make_model(summary["config"])
    model.load_state_dict(torch.load(os.path.join(RUN_CACHE, summary["key"] + ".pt"), map_location="cpu"))
    return model.to(pick_device(device)).eval()

In [ ]:
def step_time(family, device, steps=30):
    """Seconds per training step (batch 32) on one device, after 5 warm-up steps."""
    cfg = family_cfg(family, device=device)
    set_seed(0)
    dev = pick_device(device)
    model = make_model(cfg).to(dev)
    data = split_data(cfg, "train")
    opt, _, _ = make_optimizer(model, cfg, 100)
    sync = {"mps": torch.mps.synchronize, "cuda": torch.cuda.synchronize}.get(dev.type, lambda: None)
    for k, idx in enumerate(batch_indices(data, cfg["batch_size"], shuffle=True, seed=0)[:steps + 5]):
        if k == 5:
            sync()
            t = time.time()
        loss = model.loss(to_device(collate([data[i] for i in idx]), dev))[0]
        opt.zero_grad()
        loss.backward()
        opt.step()
    sync()
    return (time.time() - t) / steps

**Running a phase.** Each phase's cell below calls `run_experiment` for every run the phase needs. Cached runs load
instantly; missing runs are trained in place, one at a time, on their family's device (`device_for`: the speed benchmark's
choice, saved to `configs/devices.json`, unless `DEVICE_OVERRIDES` says otherwise). Interrupting the kernel loses only the
run in progress, so a long phase can be stopped and resumed by re-running its cell.

**Inspecting finished runs.** `cached_runs_table()` lists every finished run, or one family's with
`cached_runs_table("bilstm")`. It reads only the cached summaries, so it is safe to call at any time.

In [ ]:
def cached_runs_table(family=None):
    """One row per finished run in the run cache (dev scores, best epoch, minutes), best first within each family."""
    rows = []
    for f in sorted(os.listdir(RUN_CACHE)):
        if not f.endswith(".json") or f.startswith("hmm_"):
            continue
        with open(os.path.join(RUN_CACHE, f)) as fh:
            s = json.load(fh)
        c, d = s["config"], s["dev"]
        if family and c["family"] != family:
            continue
        knobs = (f"lr {c['lr']:g}, wd {c['weight_decay']:g}" if c["family"] == "tricrf"
                 else f"lr {c['lr']:g}, dropout {c['dropout']:g}")
        rows.append([c["family"], c.get("variant") or c["decoder"], c["emb"], knobs, s["seed"], f"{d['intent_acc']:.2f}",
                     f"{d['slot_f1']:.2f}", f"{d['exact_match']:.2f}", f"{d['combined']:.2f}", s["best_epoch"],
                     f"{s['seconds'] / 60:.0f}"])
    rows.sort(key=lambda r: (r[0], -float(r[8])))
    show_table(["family", "variant / decoder", "vectors", "recipe", "seed", "dev intent", "dev slot F1", "dev EM",
                "dev combined", "best epoch", "min"], rows)
    return len(rows)

## 9. Sanity checks before any sweep

### Parameter counts (frozen word table included)

In [ ]:
_rows = []
for _fam in REPRESENTATIVES:
    _m = make_model(family_cfg(_fam))
    _total, _table = count_system_params(_m), word_table_params(_m)
    assert _total <= MAX_PARAMS, (_fam, _total)
    _rows.append([_fam, f"{_total:,}", f"{_total - _table:,}", f"{sum(p.numel() for p in _m.parameters() if p.requires_grad):,}"])
_hmm_params = JointHMM(0.1).fit(encoded("train")).n_params
_rows.append(["hmm", f"{_hmm_params:,}", f"{_hmm_params:,}", "— (counts)"])
show_table(["family", "total params", "without word table", "trainable"], _rows)

**Recorded result** (from the run summaries; GloVe 6B 300-d word table = 1,591,200):

| Family | Total parameters | Without the word table |
|---|---|---|
| window | 1,799,755 | 208,555 |
| idcnn | 2,302,795 | 711,595 |
| textcnn | 2,445,771 | 854,571 |
| cnn_bigru | 2,699,083 | 1,107,883 |
| cnn_attn | 2,920,787 | 1,329,587 |
| transformer | 3,294,423 | 1,703,223 |
| bigru | 3,718,731 | 2,127,531 |
| bilstm | 4,398,667 | 2,807,467 |
| tricrf (sparse features + dense vectors) | 4,776,453 | 3,185,253 |
| hmm (counts) | 1,654,150 | 1,654,150 |

Every model is far below the 20,000,000 cap; only ensembles (P6) approach it. The BiLSTM variants range from 4.4M to
8.0M (V15 capsules, with 64-d slot capsules).

## 10. Phase P0: per-family recipe calibration

**Purpose:** give every family the same small tuning budget before comparing them, so no family wins because its
default learning rate happened to suit it. The recipe each family keeps here is reused unchanged in every later phase.

The cell below reloads all 54 runs from the cache (a missing run is trained in place), applies the selection rule from
section 8, prints the table and writes `configs/recipes.json`.

In [ ]:
p0 = [run_experiment(cfg, seed) for cfg, seed in p0_jobs()]
_recipes, _rows = {}, []
for _fam in REPRESENTATIVES:
    _keys = ["lr", "weight_decay"] if _fam == "tricrf" else ["lr", "dropout"]
    _runs = [s for s in p0 if s["config"]["family"] == _fam]
    _recipes[_fam] = select_recipe(_fam, [({k: s["config"][k] for k in _keys}, s["dev"]["combined"]) for s in _runs])
    for s in _runs:
        _rec = {k: s["config"][k] for k in _keys}
        _rows.append([_fam, ", ".join(f"{k} {v:g}" for k, v in _rec.items()), f"{s['params']:,}", s["best_epoch"],
                      f"{s['dev']['intent_acc']:.2f}", f"{s['dev']['slot_f1']:.2f}", f"{s['dev']['exact_match']:.2f}",
                      f"{s['dev']['combined']:.2f}", f"{s['sec_per_epoch']:.1f}", "✓" if _rec == _recipes[_fam] else ""])
with open(RECIPES_PATH, "w") as f:
    json.dump(_recipes, f, indent=1)
show_table(["family", "recipe", "params", "best epoch", "dev intent", "dev slot F1", "dev EM", "dev combined", "s/epoch", "chosen"], _rows)

**Recorded result: 49 of 54 runs** (dev, seed 0; provisional picks apply the selection rule to the runs finished so far)

| Family | Recipe | Params | Dev intent | Dev slot F1 | Dev EM | Dev combined | Best epoch | Epochs run | s / epoch | Min / run | Device | Provisional pick |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| tricrf | lr 0.001, wd 1e-05 | 4,776,453 | 86.03 | 79.68 | 67.39 | 82.85 | 16 | 26 | 19.0 | 8 | mps | ✓ |
| tricrf | lr 0.001, wd 0.0001 | 4,776,453 | 85.29 | 79.60 | 66.11 | 82.45 | 18 | 28 | 35.8 | 17 | cpu |  |
| tricrf | lr 0.003, wd 1e-05 | 4,776,453 | 85.93 | 79.13 | 66.65 | 82.53 | 6 | 16 | 34.2 | 9 | cpu |  |
| tricrf | lr 0.003, wd 0.0001 | 4,776,453 | 85.19 | 79.57 | 66.40 | 82.38 | 20 | 30 | 50.0 | 25 | cpu |  |
| tricrf | lr 0.01, wd 1e-05 | 4,776,453 | 85.49 | 79.30 | 66.75 | 82.39 | 52 | 62 | 29.3 | 30 | mps |  |
| tricrf | lr 0.01, wd 0.0001 | 4,776,453 | 85.00 | 79.73 | 66.40 | 82.36 | 32 | 42 | 34.5 | 24 | mps |  |
| window | lr 0.0005, drop 0.2 | 1,799,755 | 85.69 | 66.40 | 56.76 | 76.05 | 23 | 33 | 7.5 | 4 | cpu |  |
| window | lr 0.0005, drop 0.4 | 1,799,755 | 86.18 | 67.42 | 57.99 | 76.80 | 42 | 52 | 8.8 | 8 | cpu |  |
| window | lr 0.001, drop 0.2 | 1,799,755 | 85.64 | 67.24 | 58.24 | 76.44 | 19 | 29 | 15.3 | 7 | cpu |  |
| window | lr 0.001, drop 0.4 | 1,799,755 | 86.23 | 68.97 | 58.83 | 77.60 | 44 | 54 | 12.0 | 11 | cpu |  |
| window | lr 0.002, drop 0.2 | 1,799,755 | 85.69 | 70.82 | 59.71 | 78.25 | 41 | 51 | 14.3 | 12 | cpu | ✓ |
| window | lr 0.002, drop 0.4 | | *pending* | | | | | | | | | |
| textcnn | lr 0.0005, drop 0.2 | 2,445,771 | 86.67 | 76.44 | 66.06 | 81.56 | 55 | 65 | 7.5 | 8 | mps | ✓ |
| textcnn | lr 0.0005, drop 0.4 | 2,445,771 | 87.31 | 72.71 | 61.63 | 80.01 | 49 | 59 | 9.9 | 10 | mps |  |
| textcnn | lr 0.001, drop 0.2 | 2,445,771 | 85.64 | 76.73 | 65.03 | 81.18 | 43 | 53 | 14.8 | 13 | mps |  |
| textcnn | lr 0.001, drop 0.4 | 2,445,771 | 87.26 | 72.77 | 61.39 | 80.01 | 40 | 50 | 15.7 | 13 | mps |  |
| textcnn | lr 0.002, drop 0.2 | 2,445,771 | 86.82 | 75.60 | 63.55 | 81.21 | 57 | 67 | 9.9 | 11 | mps |  |
| textcnn | lr 0.002, drop 0.4 | 2,445,771 | 86.03 | 71.51 | 59.12 | 78.77 | 77 | 80 | 12.8 | 17 | mps |  |
| idcnn | lr 0.0005, drop 0.2 | 2,302,795 | 86.18 | 73.06 | 62.42 | 79.62 | 49 | 59 | 9.4 | 9 | mps | ✓ |
| idcnn | lr 0.0005, drop 0.4 | 2,302,795 | 85.78 | 71.90 | 61.09 | 78.84 | 80 | 80 | 18.9 | 25 | mps |  |
| idcnn | lr 0.001, drop 0.2 | 2,302,795 | 85.64 | 72.11 | 60.90 | 78.87 | 23 | 33 | 17.3 | 10 | mps |  |
| idcnn | lr 0.001, drop 0.4 | 2,302,795 | 85.10 | 66.81 | 56.47 | 75.95 | 80 | 80 | 13.5 | 18 | mps |  |
| idcnn | lr 0.002, drop 0.2 | 2,302,795 | 85.54 | 68.47 | 58.48 | 77.00 | 31 | 41 | 18.9 | 13 | mps |  |
| idcnn | lr 0.002, drop 0.4 | | *pending* | | | | | | | | | |
| bigru | lr 0.0005, drop 0.2 | 3,718,731 | 86.42 | 78.98 | 67.88 | 82.70 | 61 | 71 | 21.9 | 26 | cpu |  |
| bigru | lr 0.0005, drop 0.4 | 3,718,731 | 86.77 | 77.86 | 66.95 | 82.31 | 69 | 79 | 40.4 | 53 | cpu |  |
| bigru | lr 0.001, drop 0.2 | 3,718,731 | 86.57 | 79.26 | 67.54 | 82.91 | 52 | 62 | 34.7 | 36 | cpu | ✓ |
| bigru | lr 0.001, drop 0.4 | 3,718,731 | 86.52 | 77.15 | 66.11 | 81.84 | 29 | 39 | 70.5 | 46 | cpu |  |
| bigru | lr 0.002, drop 0.2 | 3,718,731 | 87.06 | 78.91 | 67.14 | 82.98 | 50 | 60 | 58.9 | 59 | cpu |  |
| bigru | lr 0.002, drop 0.4 | 3,718,731 | 86.77 | 77.39 | 66.36 | 82.08 | 80 | 80 | 49.9 | 67 | cpu |  |
| bilstm | lr 0.0005, drop 0.2 | 4,398,667 | 86.28 | 78.06 | 66.70 | 82.17 | 61 | 71 | 30.0 | 36 | cpu |  |
| bilstm | lr 0.0005, drop 0.4 | 4,398,667 | 86.67 | 77.42 | 66.11 | 82.04 | 48 | 58 | 53.7 | 52 | cpu |  |
| bilstm | lr 0.001, drop 0.2 | 4,398,667 | 86.57 | 78.42 | 66.80 | 82.49 | 38 | 48 | 41.7 | 33 | cpu |  |
| bilstm | lr 0.001, drop 0.4 | 4,398,667 | 86.97 | 78.57 | 67.93 | 82.77 | 52 | 62 | 71.1 | 73 | cpu | ✓ |
| bilstm | lr 0.002, drop 0.2 | 4,398,667 | 86.92 | 78.20 | 66.99 | 82.56 | 41 | 51 | 58.1 | 49 | cpu |  |
| bilstm | lr 0.002, drop 0.4 | | *pending* | | | | | | | | | |
| transformer | lr 0.00025, drop 0.1 | 3,294,423 | 85.54 | 74.84 | 63.60 | 80.19 | 35 | 45 | 19.7 | 15 | cpu |  |
| transformer | lr 0.00025, drop 0.2 | 3,294,423 | 86.42 | 75.55 | 63.99 | 80.99 | 74 | 80 | 37.4 | 50 | cpu |  |
| transformer | lr 0.0005, drop 0.1 | 3,294,423 | 86.47 | 75.25 | 64.04 | 80.86 | 53 | 63 | 27.9 | 29 | cpu |  |
| transformer | lr 0.0005, drop 0.2 | 3,294,423 | 86.18 | 75.71 | 64.19 | 80.94 | 56 | 66 | 50.8 | 56 | cpu | ✓ |
| transformer | lr 0.001, drop 0.1 | 3,294,423 | 86.23 | 74.79 | 63.21 | 80.51 | 42 | 52 | 39.4 | 34 | cpu |  |
| transformer | lr 0.001, drop 0.2 | 3,294,423 | 86.47 | 71.20 | 60.45 | 78.84 | 29 | 39 | 47.0 | 31 | cpu |  |
| cnn_bigru | lr 0.0005, drop 0.2 | 2,699,083 | 86.57 | 78.18 | 67.19 | 82.37 | 49 | 59 | 14.3 | 14 | mps |  |
| cnn_bigru | lr 0.0005, drop 0.4 | 2,699,083 | 86.62 | 75.42 | 64.73 | 81.02 | 47 | 57 | 18.3 | 17 | mps |  |
| cnn_bigru | lr 0.001, drop 0.2 | 2,699,083 | 87.16 | 78.77 | 68.13 | 82.97 | 44 | 54 | 28.1 | 25 | mps | ✓ |
| cnn_bigru | lr 0.001, drop 0.4 | 2,699,083 | 86.92 | 75.71 | 64.98 | 81.31 | 70 | 80 | 29.2 | 39 | mps |  |
| cnn_bigru | lr 0.002, drop 0.2 | 2,699,083 | 86.62 | 77.68 | 65.96 | 82.15 | 69 | 79 | 24.5 | 32 | mps |  |
| cnn_bigru | lr 0.002, drop 0.4 | 2,699,083 | 86.82 | 74.29 | 63.11 | 80.55 | 65 | 75 | 30.1 | 38 | mps |  |
| cnn_attn | lr 0.0005, drop 0.2 | 2,920,787 | 85.64 | 76.79 | 64.98 | 81.22 | 62 | 72 | 28.2 | 34 | mps |  |
| cnn_attn | lr 0.0005, drop 0.4 | 2,920,787 | 86.33 | 74.06 | 63.45 | 80.19 | 63 | 73 | 25.7 | 31 | mps |  |
| cnn_attn | lr 0.001, drop 0.2 | 2,920,787 | 86.03 | 76.85 | 65.17 | 81.44 | 62 | 72 | 25.0 | 30 | mps | ✓ |
| cnn_attn | lr 0.001, drop 0.4 | 2,920,787 | 85.64 | 73.01 | 62.08 | 79.33 | 51 | 61 | 27.0 | 27 | mps |  |
| cnn_attn | lr 0.002, drop 0.2 | | *pending* | | | | | | | | | |
| cnn_attn | lr 0.002, drop 0.4 | | *pending* | | | | | | | | | |

**Provisional recipe per family** (sorted by dev combined):

| Family | Recipes done | Provisional recipe | Dev intent | Dev slot F1 | Dev EM | Dev combined | Spread of the family's recipes (combined) |
|---|---|---|---|---|---|---|---|
| cnn_bigru | 6/6 | lr 0.001, dropout 0.2 | 87.16 | 78.77 | 68.13 | 82.97 | 80.55–82.97 (2.42) |
| bigru | 6/6 | lr 0.001, dropout 0.2 | 86.57 | 79.26 | 67.54 | 82.91 | 81.84–82.98 (1.15) |
| tricrf | 6/6 | lr 0.001, wd 1e-05 | 86.03 | 79.68 | 67.39 | 82.85 | 82.36–82.85 (0.49) |
| bilstm | 5/6 | lr 0.001, dropout 0.4 | 86.97 | 78.57 | 67.93 | 82.77 | 82.04–82.77 (0.72) |
| textcnn | 6/6 | lr 0.0005, dropout 0.2 | 86.67 | 76.44 | 66.06 | 81.56 | 78.77–81.56 (2.79) |
| cnn_attn | 4/6 | lr 0.001, dropout 0.2 | 86.03 | 76.85 | 65.17 | 81.44 | 79.33–81.44 (2.11) |
| transformer | 6/6 | lr 0.0005, dropout 0.2 | 86.18 | 75.71 | 64.19 | 80.94 | 78.84–80.99 (2.15) |
| idcnn | 5/6 | lr 0.0005, dropout 0.2 | 86.18 | 73.06 | 62.42 | 79.62 | 75.95–79.62 (3.67) |
| window | 5/6 | lr 0.002, dropout 0.2 | 85.69 | 70.82 | 59.71 | 78.25 | 76.05–78.25 (2.21) |


**What P0 shows so far**
- **Intent accuracy is nearly flat:** every one of the 49 runs scores 85.0–87.3. **Slot F1 decides the ranking:** it
  ranges from 66.4 to 79.7.
- **Four families are tied at the top** (single seed, within 0.2): CNN→BiGRU 82.97, BiGRU 82.91, TriCRF 82.85 and BiLSTM
  82.77. Separating them is P1a's job (3 seeds, paired bootstrap). Then come TextCNN 81.56, CNN→attention 81.44,
  Transformer 80.94, ID-CNN 79.62 and the window MLP 78.25.
- **The classical TriCRF has the highest slot F1** (79.1–79.7 across its six recipes): it is the only family with a CRF
  in P0, which matches Gilbert's finding that a CRF adds about 4 slot-F1 points. P3 adds the CRF to every neural family.
- **Dropout 0.4 lowers slot F1** at the same learning rate by 1.2–5.3 points in the convolutional families and by
  1.1–2.1 in BiGRU, but not in BiLSTM, whose best run uses 0.4.
- **Recipe sensitivity differs by family:** TriCRF's six recipes span 0.49 points and BiLSTM's 0.72, but ID-CNN's span
  3.67 and TextCNN's 2.79. For the flat families the exact pick barely matters.
- **Grid edges and the epoch cap:** the window MLP's best uses the highest learning rate; TextCNN's, ID-CNN's and the
  Transformer's best use the lowest; three runs peaked at the 80-epoch cap (ID-CNN twice, BiGRU once) and two more
  after epoch 74 (Transformer, TextCNN). Better recipes may lie just outside the grid for these families.
- **The Transformer agrees with Gilbert's notebook:** 86.18 / 75.71 here (no CRF, 1 seed) vs his 86.90 / 76.09 (3 seeds).
- **Cost:** the RNN and Transformer runs took 15–73 minutes each on this laptop, the other families 4–39.

**Pending:** the 5 runs marked *pending* (all at lr 2e-3), then this cell writes `configs/recipes.json`.

### HMM smoothing (P0, R0)

Four deterministic fits, seconds each; the best k joins the recipes.

In [ ]:
_hmm = [run_hmm(k) for k in HMM_K]
show_table(["HMM add-k", "dev intent", "dev slot F1", "dev EM", "dev combined"],
           [[r["config"]["k"], f"{r['dev']['intent_acc']:.2f}", f"{r['dev']['slot_f1']:.2f}", f"{r['dev']['exact_match']:.2f}",
             f"{r['dev']['combined']:.2f}"] for r in _hmm])
_recipes["hmm"] = {"k": max(_hmm, key=lambda r: r["dev"]["combined"])["config"]["k"]}
with open(RECIPES_PATH, "w") as f:
    json.dump(_recipes, f, indent=1)
print("recipes fixed for every later phase:", _recipes)

**Recorded result:** *pending* (runs after the P0 cell above).

### Cross-check of the shared metric against `seqeval`
On real predictions (the best P0 BiLSTM, dev), our multiset (type, text) F1 must agree with `seqeval`'s span F1
within 0.2 points. They can differ only when the same slot text occurs twice in one utterance.

In [ ]:
from seqeval.metrics import f1_score as seqeval_f1

_best = max((s for s in p0 if s["config"]["family"] == "bilstm"), key=lambda s: s["dev"]["combined"])
_res, (_names, _spans) = evaluate(load_run_model(_best), _best["config"], "dev")
_gold = [spans_to_tags(len(ex["tokens"]), ex["spans"]) for ex in examples["dev"]]
_pred = [spans_to_tags(len(ex["tokens"]), sp) for ex, sp in zip(examples["dev"], _spans)]
_sq = 100 * seqeval_f1(_gold, _pred)
print(f"shared metric {_res['slot_f1']:.2f} vs seqeval {_sq:.2f}")
assert abs(_sq - _res["slot_f1"]) < 0.2

**Recorded result:** *pending* (needs the finished P0 BiLSTM runs).

## 11. Analysis helpers
Every comparison below uses these:
- **`utterance_stats`** rebuilds per-utterance intent hits and slot TP / FP / FN from a run's saved dev predictions.
  The cell after the helpers checks that they add up to every logged P0 score.
- **`paired_bootstrap`** resamples the 2,033 dev utterances 1,000 times and recomputes the seed-averaged difference
  between two configurations on each resample, giving a 95% interval (Berg-Kirkpatrick et al. 2012). Slot F1 is
  recomputed from the pooled TP / FP / FN of each resample, not averaged per utterance.
- **`two_way_anova`** splits a grid of 3-seed cells into row, column and interaction effects and tests each against
  seed-to-seed noise.
- **`kendall`** asks whether the rows rank the columns the same way: τ-b for each pair of rows, Kendall's W for all.
- **`error_profile`** sorts slot errors into boundary, type, spurious and missed spans, splits recall by whether the
  gold span contains a word unseen in train, and counts intent errors that stay inside the gold scenario.

Charts use one blue ramp for magnitudes and a blue ↔ red scale centred on zero for differences (blue = higher).

In [ ]:
# analysis helpers
from scipy import stats
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

METRICS = {"intent_acc": "intent acc", "slot_f1": "slot F1", "exact_match": "exact match", "combined": "combined"}


def mean_sd(runs, metric="combined"):
    v = np.array([r["dev"][metric] for r in runs])
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)


def fmt_ms(runs, metric="combined"):
    m, s = mean_sd(runs, metric)
    return f"{m:.2f} ± {s:.2f}"


def utterance_stats(run, exs=None):
    """Per-utterance dev arrays [4, n] rebuilt from a run's saved predictions: intent correct, slot TP, FP, FN."""
    exs = examples["dev"] if exs is None else exs
    pred = run["dev_pred"]
    out = np.zeros((4, len(exs)))
    for i, (ex, pi, ps) in enumerate(zip(exs, pred["intents"], pred["spans"], strict=True)):
        gold, hyp = slot_pairs(ex["tokens"], ex["spans"]), slot_pairs(ex["tokens"], [tuple(s) for s in ps])
        hit = sum((gold & hyp).values())
        out[:, i] = (pi == ex["intent"], hit, sum(hyp.values()) - hit, sum(gold.values()) - hit)
    return out


def _seed_mean_scores(w, S):
    """w [R, n] utterance weights (resample counts); S [seeds, 4, n] -> seed-averaged intent acc and slot F1 per row of w."""
    tot = np.einsum("rn,skn->rsk", w, S)
    acc = 100 * tot[..., 0] / w.sum(1, keepdims=True)
    f1 = 100 * 2 * tot[..., 1] / np.maximum(2 * tot[..., 1] + tot[..., 2] + tot[..., 3], 1)
    return acc.mean(1), f1.mean(1)


def paired_bootstrap(runs_a, runs_b, n_boot=1000, seed=0):
    """Paired bootstrap over dev utterances of A − B on seed-averaged scores.
    Returns {metric: (observed difference, 2.5th percentile, 97.5th percentile)}."""
    A = np.stack([utterance_stats(r) for r in runs_a])
    B = np.stack([utterance_stats(r) for r in runs_b])
    n = A.shape[-1]
    w = np.random.default_rng(seed).multinomial(n, np.full(n, 1 / n), size=n_boot).astype(float)
    out = {}
    for name, ww in (("obs", np.ones((1, n))), ("boot", w)):
        (ia, fa), (ib, fb) = _seed_mean_scores(ww, A), _seed_mean_scores(ww, B)
        out[name] = {"intent_acc": ia - ib, "slot_f1": fa - fb, "combined": (ia + fa - ib - fb) / 2}
    return {k: (out["obs"][k][0], *np.percentile(out["boot"][k], [2.5, 97.5])) for k in out["obs"]}


def two_way_anova(Y):
    """Y [rows, cols, seeds]. Two-way ANOVA with replication: grand mean, row and column effects, interaction residuals,
    and a table of (source, SS, df, F, p, share of the total sum of squares)."""
    R, C, S = Y.shape
    cell = Y.mean(-1)
    mu = cell.mean()
    row, col = cell.mean(1) - mu, cell.mean(0) - mu
    inter = cell - mu - row[:, None] - col[None, :]
    ss = {"rows": C * S * (row ** 2).sum(), "columns": R * S * (col ** 2).sum(), "interaction": S * (inter ** 2).sum(),
          "seeds (residual)": ((Y - cell[..., None]) ** 2).sum()}
    df = {"rows": R - 1, "columns": C - 1, "interaction": (R - 1) * (C - 1), "seeds (residual)": R * C * (S - 1)}
    ms_res, total = ss["seeds (residual)"] / df["seeds (residual)"], sum(ss.values())
    table = []
    for k in ss:
        F = (ss[k] / df[k]) / ms_res if k != "seeds (residual)" else np.nan
        p = stats.f.sf(F, df[k], df["seeds (residual)"]) if k != "seeds (residual)" else np.nan
        table.append((k, ss[k], df[k], F, p, 100 * ss[k] / total))
    return {"mu": mu, "row": row, "col": col, "inter": inter, "table": table, "noise_sd": np.sqrt(ms_res)}


def kendall(cell):
    """cell [rows, cols]: Kendall's τ-b between every pair of rows' rankings of the columns, and Kendall's W
    (coefficient of concordance) over all rows with its χ² p-value."""
    R, C = cell.shape
    tau = np.eye(R)
    for i in range(R):
        for j in range(i + 1, R):
            tau[i, j] = tau[j, i] = stats.kendalltau(cell[i], cell[j]).statistic
    rank_sums = np.stack([stats.rankdata(r) for r in cell]).sum(0)
    W = 12 * ((rank_sums - rank_sums.mean()) ** 2).sum() / (R ** 2 * (C ** 3 - C))
    return tau, W, stats.chi2.sf(R * (C - 1) * W, C - 1)


def error_profile(runs, exs=None):
    """Slot and intent error anatomy on dev, pooled over a configuration's seeds (position-based, diagnosis only)."""
    exs = examples["dev"] if exs is None else exs
    w2i, c = vocab["word2id"], collections.Counter()
    for run in runs:
        for ex, pi, ps in zip(exs, run["dev_pred"]["intents"], run["dev_pred"]["spans"], strict=True):
            gold, pred = [tuple(s) for s in ex["spans"]], [tuple(s) for s in ps]
            for g in gold:
                kind = "oov" if any(t not in w2i for t in ex["tokens"][g[1]:g[2]]) else "iv"
                c["gold_" + kind] += 1
                c["hit_" + kind] += g in pred
                c["missed"] += not any(g[1] < p[2] and p[1] < g[2] for p in pred)
            for p in pred:
                over = [g for g in gold if g[1] < p[2] and p[1] < g[2]]
                c["pred"] += 1
                c["exact" if p in gold else "type" if any(g[1:] == p[1:] for g in over) else
                  "boundary" if any(g[0] == p[0] for g in over) else "type+boundary" if over else "spurious"] += 1
            if pi != ex["intent"]:
                c["intent_err"] += 1
                c["intent_err_same_scenario"] += pi.split("_")[0] == ex["intent"].split("_")[0]
    return c


# chart style: the dataviz reference palette (one-hue sequential ramp; blue ↔ red diverging with a grey midpoint)
SEQ = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
       "#184f95", "#104281", "#0d366b"]
DIV = ["#1c5cab", "#3987e5", "#9ec5f4", "#f0efec", "#f3b3b2", "#e34948", "#a8302f"]
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, MUTED, GRID, AXIS, SURFACE, GREY = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb", "#b9b8b2"
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.titlecolor": INK,
    "axes.grid": False, "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "axes.prop_cycle": mpl.cycler(color=CAT), "legend.frameon": False, "figure.dpi": 110,
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "axes.titlesize": 10, "axes.titlelocation": "left", "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
})


def heatmap(ax, M, rows, cols, title, fmt="{:.1f}", diverging=False, label=""):
    """Annotated grid: sequential blue for magnitudes, or diverging blue/red centred on 0 for differences."""
    if diverging:
        lim = float(np.nanmax(np.abs(M))) or 1.0
        cmap, norm = LinearSegmentedColormap.from_list("div", DIV[::-1]), TwoSlopeNorm(0.0, -lim, lim)
    else:
        cmap, norm = LinearSegmentedColormap.from_list("seq", SEQ), None
    im = ax.imshow(M, cmap=cmap, norm=norm, aspect="auto")
    for i, j in np.ndindex(M.shape):
        r, g, b, _ = im.cmap(im.norm(M[i, j]))
        ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=7.5,
                color="#ffffff" if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.45 else INK)
    ax.set_xticks(range(len(cols)), cols, rotation=30, ha="right")
    ax.set_yticks(range(len(rows)), rows)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title)
    cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cb.outline.set_visible(False)
    cb.ax.tick_params(length=0, labelsize=7, colors=INK2)
    if label:
        cb.set_label(label, color=INK2, fontsize=8)
    return im


def label_points(ax, xs, ys, names):
    for x, y, n in zip(xs, ys, names):
        ax.annotate(n, (x, y), xytext=(0, 6), textcoords="offset points", ha="center", fontsize=7.5, color=INK2)


def pareto_mask(cost, score):
    """True where no other point is at least as cheap and strictly better, or strictly cheaper and at least as good."""
    cost, score = np.asarray(cost), np.asarray(score)
    return np.array([not np.any(((cost <= c) & (score > s)) | ((cost < c) & (score >= s))) for c, s in zip(cost, score)])

In [ ]:
_n_checked = 0
for _s in p0:
    if "dev_pred" in _s:
        _S = utterance_stats(_s)
        assert int(_S[0].sum()) == _s["dev"]["n_correct_intent"], _s["key"]
        assert tuple(int(x) for x in _S[1:].sum(1)) == (_s["dev"]["tp"], _s["dev"]["fp"], _s["dev"]["fn"]), _s["key"]
        _n_checked += 1
print(f"utterance_stats reproduces the logged dev counts of {_n_checked} P0 runs")

**Recorded result:** the helpers were checked on 4 November 2026 outside the notebook. `utterance_stats` reproduced the
logged dev counts (TP / FP / FN and correct intents) of the P0 runs exactly; the bootstrap's observed difference equalled
the logged difference; the ANOVA's sums of squares added up to the total, with 3.7% false positives at p < 0.05 under a
null model (nominal 5%); Kendall's τ matched `scipy`. All P1a cells below then ran end to end on a mock 189-run grid.

## 12. Phase P1a: the baseline × embedding grid
Nine family representatives × seven vector sources × 3 seeds = 189 runs, each family with its P0 recipe.
Pretrained tables are frozen (P2 fine-tunes the best three); `random` is a trainable N(0, 0.4²) table, and for the
triangular CRF it means sparse features only. Cached runs reload instantly; a missing run is trained here.

**Status:** not started (189 runs). The cells below are ready and were tested on a mock grid.

In [ ]:
p1a = collections.defaultdict(list)
for _cfg, _seed in p1a_jobs():
    p1a[_cfg["family"], _cfg["emb"]].append(run_experiment(_cfg, _seed))
FAMS, EMBS = REPRESENTATIVES, EMBEDDING_NAMES
Y1 = {m: np.array([[[r["dev"][m] for r in p1a[f, e]] for e in EMBS] for f in FAMS]) for m in METRICS}
for _m in ("combined", "intent_acc", "slot_f1", "exact_match"):
    _best = Y1[_m].mean(-1).argmax(1)
    print(f"\nDev {METRICS[_m]}, mean ± sd over 3 seeds (each family's best source in bold)")
    show_table(["family"] + EMBS, [[f] + [("**%s**" if j == _best[i] else "%s") % fmt_ms(p1a[f, e], _m)
                                         for j, e in enumerate(EMBS)] for i, f in enumerate(FAMS)])

In [ ]:
_fig, _axes = plt.subplots(1, 3, figsize=(17, 4.6))
for _ax, _m in zip(_axes, ("intent_acc", "slot_f1", "combined")):
    heatmap(_ax, Y1[_m].mean(-1), FAMS, EMBS, f"Dev {METRICS[_m]}, mean of 3 seeds", label=METRICS[_m])
_fig.tight_layout()
plt.show()

### How much do the baseline, the embedding and their interaction matter?
A two-way ANOVA with 3 seeds per cell splits each grid into a baseline effect (row mean − grand mean), an embedding
effect (column mean − grand mean) and interaction residuals, and tests each against the seed-to-seed noise. The share
of variance says how much of the spread in scores each factor explains. A large interaction would mean the best
embedding depends on the architecture.

In [ ]:
ANOVA1, _rows = {}, []
for _m in ("intent_acc", "slot_f1", "combined"):
    ANOVA1[_m] = two_way_anova(Y1[_m])
    for _src, _ss, _df, _F, _p, _share in ANOVA1[_m]["table"]:
        _rows.append([METRICS[_m], {"rows": "baseline", "columns": "embedding"}.get(_src, _src), f"{_ss:.1f}", _df,
                      "" if np.isnan(_F) else f"{_F:.1f}", "" if np.isnan(_p) else f"{_p:.1e}", f"{_share:.1f}%"])
show_table(["metric", "source", "sum of squares", "df", "F", "p", "share of variance"], _rows)
_A = ANOVA1["combined"]
print(f"grand mean {_A['mu']:.2f}; seed noise sd {_A['noise_sd']:.2f}, so a 3-seed cell mean has SE {_A['noise_sd'] / np.sqrt(3):.2f}")
show_table(["family", "baseline effect (combined)"], [[f, f"{v:+.2f}"] for f, v in sorted(zip(FAMS, _A["row"]), key=lambda t: -t[1])])
show_table(["embedding", "embedding effect (combined)"], [[e, f"{v:+.2f}"] for e, v in sorted(zip(EMBS, _A["col"]), key=lambda t: -t[1])])
_fig, _ax = plt.subplots(figsize=(7.5, 4.6))
heatmap(_ax, _A["inter"], FAMS, EMBS, "Interaction residuals, dev combined (cell − baseline − embedding effect)",
        fmt="{:+.2f}", diverging=True, label="points")
plt.show()

### Do the families agree on which embedding is best?
Kendall's τ-b compares two families' rankings of the seven sources (+1 identical, 0 unrelated, −1 reversed);
Kendall's W summarises agreement across all nine (1 = everyone ranks them the same).

In [ ]:
_tau, _W, _p = kendall(Y1["combined"].mean(-1))
print(f"Kendall's W over the 9 families: {_W:.2f} (χ² p = {_p:.1e}); mean pairwise τ {(_tau.sum() - len(FAMS)) / (len(FAMS) * (len(FAMS) - 1)):.2f}")
_fig, _ax = plt.subplots(figsize=(7, 5.2))
heatmap(_ax, _tau, FAMS, FAMS, "Kendall's τ between families' rankings of the 7 sources (dev combined)", fmt="{:+.2f}",
        diverging=True, label="τ")
plt.show()

### Embedding contrasts built into the grid
- **pretrained − random:** the value of pretraining (mean of the six sources minus the random table);
- **fastText − GloVe** and **Common Crawl − Wikipedia/news:** the two main effects of the algorithm × corpus 2 × 2
  (GloVe 6B / 42B and fastText wiki-news / crawl);
- **algorithm × corpus:** how much more the corpus matters for fastText than for GloVe;
- **window − dependency:** word2vec GoogleNews minus Komninos (same algorithm family, different contexts).

The last row gives each contrast's standard error from the seed noise (one family).

In [ ]:
def _contrasts(cell):
    v = dict(zip(EMBS, cell))
    return [np.mean([v[e] for e in EMBS if e != "random"]) - v["random"],
            (v["ft_wiki"] + v["ft_crawl"] - v["glove6b"] - v["glove42b"]) / 2,
            (v["glove42b"] + v["ft_crawl"] - v["glove6b"] - v["ft_wiki"]) / 2,
            (v["ft_crawl"] - v["ft_wiki"]) - (v["glove42b"] - v["glove6b"]),
            v["w2v"] - v["komninos"]]


_names = ["pretrained − random", "fastText − GloVe", "Common Crawl − Wiki/news", "algorithm × corpus", "window − dependency"]
_se_w = np.sqrt(np.array([7 / 6, 1, 1, 4, 2]) / 3)                  # √(Σ weights² / seeds) for each contrast
for _m in ("combined", "intent_acc", "slot_f1"):
    _C = np.array([_contrasts(c) for c in Y1[_m].mean(-1)])
    print(f"\nEmbedding contrasts, dev {METRICS[_m]} (points)")
    show_table(["family"] + _names, [[f] + [f"{x:+.2f}" for x in row] for f, row in zip(FAMS, _C)]
               + [["**mean of families**"] + [f"**{x:+.2f}**" for x in _C.mean(0)]]
               + [["SE (one family)"] + [f"{x:.2f}" for x in ANOVA1[_m]["noise_sd"] * _se_w]])

In [ ]:
_info = {e: embedding_matrix(e)[1] for e in EMBEDDINGS}
_x = [_info[e]["missing_pct"] for e in EMBEDDINGS]
_y = [ANOVA1["combined"]["col"][EMBS.index(e)] for e in EMBEDDINGS]
_fig, _ax = plt.subplots(figsize=(6, 3.8))
_ax.grid(True, axis="y")
_ax.scatter(_x, _y, s=40, color=CAT[0], edgecolors=SURFACE, linewidths=2, zorder=3)
label_points(_ax, _x, _y, list(EMBEDDINGS))
_ax.set_xlabel("train word types missing from the source (%)")
_ax.set_ylabel("embedding effect, dev combined (points)")
_ax.set_title("Coverage vs embedding effect (pretrained sources)")
plt.show()

### Ranking the families at their best embedding
Each family at the source with its best 3-seed mean. Δ is the paired-bootstrap difference from the top configuration
with its 95% interval; families whose interval covers 0 are statistically tied with it on dev.

In [ ]:
BEST1 = {f: EMBS[int(Y1["combined"][i].mean(-1).argmax())] for i, f in enumerate(FAMS)}
_order = sorted(FAMS, key=lambda f: -mean_sd(p1a[f, BEST1[f]])[0])
_top = p1a[_order[0], BEST1[_order[0]]]
_rows = []
for _k, _f in enumerate(_order, 1):
    _runs = p1a[_f, BEST1[_f]]
    if _k == 1:
        _delta, _tie = "—", "best"
    else:
        _d, _lo, _hi = paired_bootstrap(_runs, _top)["combined"]
        _delta, _tie = f"{_d:+.2f} [{_lo:+.2f}, {_hi:+.2f}]", "tied" if _lo <= 0 <= _hi else ("worse" if _hi < 0 else "better")
    _rows.append([_k, _f, BEST1[_f], fmt_ms(_runs, "intent_acc"), fmt_ms(_runs, "slot_f1"), fmt_ms(_runs, "exact_match"),
                  fmt_ms(_runs), _delta, _tie])
show_table(["rank", "family", "best source", "intent acc", "slot F1", "exact match", "combined", "Δ combined vs best [95% CI]", ""], _rows)

### Efficiency
Parameters with and without the word table, time per epoch on the device each family trains on (with 4 workers
sharing the machine), epochs to the best dev score, and minutes per run. The Pareto front (highlighted) holds the
families that no other family beats on both score and cost.

In [ ]:
_rows, _pts = [], []
for _f in FAMS:
    _runs = p1a[_f, BEST1[_f]]
    _p, _wt = _runs[0]["params"], _runs[0]["word_table_params"]
    _spe, _ep, _min = (np.mean([r["sec_per_epoch"] for r in _runs]), np.mean([r["best_epoch"] for r in _runs]),
                       np.mean([r["seconds"] for r in _runs]) / 60)
    _rows.append([_f, f"{_p:,}", f"{_p - _wt:,}", f"{_spe:.1f}", f"{_ep:.1f}", f"{_min:.1f}", _runs[0]["config"]["device"]])
    _pts.append((_p - _wt, _min, mean_sd(_runs)[0]))
show_table(["family", "params", "without word table", "s / epoch", "best epoch", "min / run", "device"], _rows)
_fig, _axes = plt.subplots(1, 2, figsize=(13, 4))
for _ax, _j, _xl in zip(_axes, (0, 1), ("parameters without the word table (log scale)", "minutes per run (log scale)")):
    _c, _s = [p[_j] for p in _pts], [p[2] for p in _pts]
    _front = pareto_mask(_c, _s)
    _ax.grid(True, axis="y")
    _ax.scatter(_c, _s, s=40, color=[CAT[0] if f else GREY for f in _front], edgecolors=SURFACE, linewidths=2, zorder=3)
    label_points(_ax, _c, _s, FAMS)
    _ax.set_xscale("log")
    _ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(
        (lambda v, _: f"{v / 1e6:g}M") if _j == 0 else (lambda v, _: f"{v:g}")))
    _ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    _ax.set_xlabel(_xl)
    _ax.set_ylabel("dev combined, best source")
    _ax.set_title("Score vs " + ("size" if _j == 0 else "training time") + " (blue = Pareto front)")
_fig.tight_layout()
plt.show()

### Learning curves (seed 0, each family at its best source)
Top: training loss (the family's own joint objective, so scales differ between panels) and dev loss.
Bottom: dev intent accuracy and slot F1. The dot marks the best dev epoch, which the checkpoint keeps.

In [ ]:
_fig, _axes = plt.subplots(2, len(FAMS), figsize=(2.3 * len(FAMS), 5.2), sharex=False)
for _j, _f in enumerate(FAMS):
    _h, _be = p1a[_f, BEST1[_f]][0]["history"], p1a[_f, BEST1[_f]][0]["best_epoch"]
    _ep = [h["epoch"] for h in _h]
    _a, _b = _axes[0, _j], _axes[1, _j]
    _a.plot(_ep, [h["train_loss"] for h in _h], color=CAT[0], label="train loss")
    _a.plot(_ep, [h["dev_loss"] for h in _h], color=CAT[1], label="dev loss")
    _b.plot(_ep, [h["dev_intent_acc"] for h in _h], color=CAT[0], label="dev intent acc")
    _b.plot(_ep, [h["dev_slot_f1"] for h in _h], color=CAT[1], label="dev slot F1")
    _b.scatter([_be] * 2, [_h[_be - 1]["dev_intent_acc"], _h[_be - 1]["dev_slot_f1"]], s=30, color=INK, zorder=3)
    _a.set_title(_f)
    _b.set_xlabel("epoch")
    _b.set_ylim(50, 92)
    for _x in (_a, _b):
        _x.grid(True, axis="y")
_axes[0, 0].legend(fontsize=7, loc="upper right")
_axes[1, 0].legend(fontsize=7, loc="lower right")
_fig.tight_layout()
plt.show()

### Error profiles (dev, pooled over the 3 seeds, each family at its best source)
Predicted spans are exact, a type error (right extent, wrong type), a boundary error (right type, overlapping but
wrong extent), both, or spurious (no overlap with any gold span). Missed = gold spans with no overlapping prediction.
OOV recall covers gold spans containing a word unseen in train; in-vocab recall covers the rest.

In [ ]:
_rows = []
for _f in FAMS:
    _c = error_profile(p1a[_f, BEST1[_f]])
    _wrong = _c["pred"] - _c["exact"]
    _rows.append([_f, f"{100 * _c['exact'] / _c['pred']:.1f}", *(f"{100 * _c[k] / max(_wrong, 1):.0f}%" for k in
                  ("boundary", "type", "type+boundary", "spurious")), f"{_c['missed'] / 3:.0f}",
                  f"{100 * _c['hit_iv'] / _c['gold_iv']:.1f}", f"{100 * _c['hit_oov'] / _c['gold_oov']:.1f}",
                  f"{100 * _c['intent_err_same_scenario'] / max(_c['intent_err'], 1):.0f}%"])
show_table(["family", "span precision", "wrong spans: boundary", "type", "both", "spurious", "missed spans / seed",
            "in-vocab recall", "OOV recall", "intent errors within scenario"], _rows)

## 13. Phase P1b: BiLSTM decoder and interaction variants

The 17 variants share the BiLSTM's embedder, encoder and P0 recipe, and each changes one thing: the slot decoder
(V1–V8) or how intent and slots exchange information (V9–V17). Each runs with the BiLSTM's best P1a source and with
the random table, 3 seeds each. Δ is the paired-bootstrap difference from the plain BiLSTM under the same source and
seeds (P1a), with its 95% interval. The joint HMM, which has no embedding, appears once at the bottom.
V15 (capsules) did not memorise the intents in the smoke test (section 9) and is marked ⚠.

**Status:** not started (102 runs + the HMM). Code ready.

In [ ]:
P1B_EMBS = sorted({best_embedding("bilstm"), "random"}, key=EMBEDDING_NAMES.index)
p1b = {(v, e): [run_experiment(rep_cfg("bilstm", emb=e, **sp), s) for s in SEEDS]
       for v, sp in VARIANT_SPECS.items() for e in P1B_EMBS}
_hmm_run = run_hmm(recipe("hmm")["k"])
for _e in P1B_EMBS:
    _base = p1a["bilstm", _e]
    _rows = [["—", "plain BiLSTM (P1a)", fmt_ms(_base, "intent_acc"), fmt_ms(_base, "slot_f1"), fmt_ms(_base, "exact_match"),
              fmt_ms(_base), "", f"{_base[0]['params']:,}", f"{np.mean([r['sec_per_epoch'] for r in _base]):.1f}"]]
    for _v, _sp in VARIANT_SPECS.items():
        _runs = p1b[_v, _e]
        _d, _lo, _hi = paired_bootstrap(_runs, _base)["combined"]
        _name = _sp.get("variant", f"crf-{_sp.get('scheme', 'BIO')}") + (" ⚠" if _v == "V15" else "")
        _rows.append([_v, _name, fmt_ms(_runs, "intent_acc"), fmt_ms(_runs, "slot_f1"), fmt_ms(_runs, "exact_match"),
                      fmt_ms(_runs), f"{_d:+.2f} [{_lo:+.2f}, {_hi:+.2f}]" + (" *" if _lo > 0 or _hi < 0 else ""),
                      f"{_runs[0]['params']:,}", f"{np.mean([r['sec_per_epoch'] for r in _runs]):.1f}"])
    print(f"\nSource: {_e}  (* = interval excludes 0)")
    show_table(["id", "variant", "intent acc", "slot F1", "exact match", "combined", "Δ combined vs plain [95% CI]",
                "params", "s / epoch"], _rows)
_h = _hmm_run["dev"]
print(f"\nJoint HMM (add-k {recipe('hmm')['k']}, {_hmm_run['params']:,} counts): intent {_h['intent_acc']:.2f}, "
      f"slot F1 {_h['slot_f1']:.2f}, exact match {_h['exact_match']:.2f}, combined {_h['combined']:.2f}")

### Which variants help, and does the answer depend on the vectors?

A diverging heatmap of each variant minus the plain BiLSTM, and a two-way ANOVA (variant × source).

In [ ]:
_V = list(VARIANT_SPECS)
_D = np.array([[[r["dev"]["combined"] for r in p1b[v, e]] for e in P1B_EMBS] for v in _V])
_P = np.array([[r["dev"]["combined"] for r in p1a["bilstm", e]] for e in P1B_EMBS])
_fig, _ax = plt.subplots(figsize=(6.5, 6.5))
heatmap(_ax, _D.mean(-1) - _P.mean(-1)[None, :],
        [f"{v} {VARIANT_SPECS[v].get('variant', 'crf-' + VARIANT_SPECS[v].get('scheme', 'BIO'))}" for v in _V], P1B_EMBS,
        "Variant − plain BiLSTM, dev combined (points)", fmt="{:+.2f}", diverging=True, label="points")
plt.show()
_A = two_way_anova(_D)
show_table(["source", "sum of squares", "df", "F", "p", "share of variance"],
           [[{"rows": "variant", "columns": "embedding"}.get(s, s), f"{ss:.1f}", df, "" if np.isnan(F) else f"{F:.1f}",
             "" if np.isnan(p) else f"{p:.1e}", f"{sh:.1f}%"] for s, ss, df, F, p, sh in _A["table"]])

## 14. Phases P2–P7 (planned; code not written yet)

| Phase | Design | Question it answers |
|---|---|---|
| **P2** frozen vs fine-tuned | Each representative × the top-3 pretrained sources from P1a, fine-tuned at LR × 0.1, 3 seeds (81 runs), paired with the frozen P1a runs | Does adapting the vectors help, and for which families? (Expect small gains for RNN/Transformer, possible losses for window/CNN.) |
| **P3** techniques in isolation and combined | 8 neural representatives × {+CRF (λ 0.7), +char-CNN, +word dropout (α 0.75), +label smoothing (0.1), +all four} × 3 seeds; 17 variants × {+all applicable} × 3 seeds (171 runs) | Which technique helps which family, and does the all-four package close the gaps between families? |
| **P4** extensions on the best baseline | char-BiLSTM, train-only BPE, shape flags, possessive backoff, word-dropout α ∈ {0.25, 1}, λ ∈ {0.5, 0.85}, supervised contrastive loss, boundary auxiliaries, R-Drop, EMA, self-distillation (39 runs) | What else helps the strongest model? |
| **P5** combination | 2^(5−1) resolution-V fractional factorial on the five strongest techniques (16 cells × 3 seeds), then a short random search (72 runs) | Which techniques reinforce or cancel each other? |
| **P6** ensembles within 20M | K seeds of the best configuration vs mixed families vs mixed vector sources; intent probabilities averaged, CRF scores summed (product of experts) | Does diversity beat more seeds at equal parameter budget? |
| **P7** final system | Retrain the frozen configuration with fresh seeds (100+), log per-epoch train/dev loss, dev metrics, elapsed time and learning curves; save the best-dev checkpoint (epoch, score, filename); reload it and assert the dev scores match; then evaluate test **once** for the final system and each family's best configuration | The unbiased test estimate |

The final-system section will also print and assert the system's total parameter count and report per-slot-type F1,
intent confusions and OOV vs in-vocabulary slot recall.

## 15. Results so far and what is left

### Results so far (dev only; test untouched)

| Result | Value |
|---|---|
| Splits | 11,514 / 2,033 / 2,974 asserted; annotation round trip exact on all 16,521 rows |
| Vocabularies (train only) | 5,304 words, 34 characters, 60 intents, 55 slot types; BIO 111 / BIOES 221 / IO 56 tags |
| Dev OOV | 3.32% of tokens |
| Vector coverage (train types missing) | GloVe 42B 1.49%, fastText crawl 1.68%, fastText wiki 2.64%, word2vec 2.68%, GloVe 6B 3.47%, Komninos 4.77% |
| Correctness checks | metric unit tests, CRF and semi-CRF brute force, constrained Viterbi, padding invariance: all pass |
| Memorise 64 utterances | all 9 representatives and 16 of 17 variants pass; V15 capsules memorise slots (100.00) but not intents (50.00) |
| Largest single model | 8.0M parameters (V15); every representative ≤ 4.8M, so the 20M cap binds only ensembles |
| **P0 (49 / 54 runs)** | best per family (dev combined): CNN→BiGRU 82.97, BiGRU 82.91, TriCRF 82.85, BiLSTM 82.77, TextCNN 81.56, CNN→attention 81.44, Transformer 80.94, ID-CNN 79.62, window 78.25 |

**Reference points**

| System | Dev intent | Dev slot F1 | Test intent | Test slot F1 |
|---|---|---|---|---|
| Gilbert's Transformer, no CRF (3 seeds) | 86.90 | 76.09 | — | — |
| Gilbert's Transformer + CRF, λ 0.5 (his final configuration) | 85.87 | 80.28 | 84.47 | 77.55 |
| Our P0 best single runs, no CRF except the TriCRF | 85.0–87.3 | up to 79.68 | — | — |
| XLM-R Base (pretrained; not allowed here) | — | — | 88.3 | 78.7 |

### Issues found and fixed along the way

| Issue | Effect | Fix |
|---|---|---|
| `device_for` read the device plan from the wrong JSON level | Every P0 job went to MPS and ran slowly | Fixed; the cache key now ignores the device |
| MPS convolution backward aborts on all-one-token batches | Killed every CNN→attention run (the pool then waited forever for the lost job) | `collate` pads to ≥ 2 positions; padding invariance re-checked |
| Capsule routing starved 111 tag capsules | V15 could not learn | Routed sum scaled by n_out / n_in, LayerNorm, 64-d capsules; intents still not memorised (flagged) |
| Recipe tie-break compared floating-point distances | BiLSTM would have kept dropout 0.2 over the tied, higher-scoring 0.4 | Distances rounded (found while writing this notebook) |
| OneDrive evicted project files to the cloud (disk 95% full) | Background training workers could not read the code and respawned in a loop, loading the machine | Run stopped; files restored and verified; caches kept outside OneDrive |

### What is left

1. **Finish P0:** re-running the P0 cell trains the 5 pending runs (all at lr 2e-3; about 1–2 hours on this laptop),
   then writes `configs/recipes.json`; the HMM smoothing is chosen and the `seqeval` cross-check runs.
2. **Decide where the heavy runs happen.** This laptop managed about 7 runs per hour, and RNN/Transformer runs take
   30–70 minutes, so P1a alone is roughly a day of heavy load and the full plan several days. The options are a GPU
   elsewhere (the notebook also runs on CUDA, e.g. Colab or an NTU server: about 1–3 minutes per run), or a lighter
   protocol here (batch 64, ≤ 40 epochs, patience 5, 2 seeds for screening).
3. **Run P1a and P1b**, whose analysis cells are ready (sections 12–13).
4. **Write and run P2–P6**, then **P7**: the final system with live training logs, checkpoint reload check, and the single
   test evaluation.